In [ ]:
# All important libaries to have installed and ignore installation if the lib exists
# requests beautifulsoup4 lxml pymupdf pdfplumber pandas numpy sentence-transformers chromadb Flask flask-cors pydantic

    1. One time Scraping

    Scraping the Kenya Law website

Scrape the Kenya law website and get all the required data to be used in the RAG system and save them in a folder called
scraped_raw_pdfs each with its folder insiide this folder e.g scraped_raw_pdfs/constitution, scraped_raw_pdfs/all_legislations where
the files are located in their respective directories. For testing we will do a maximum of 5to make sure everything runs correctly
before moving to the production scraping.

In [ ]:
# BASE FILE CONFIGURATION

import os
import json
import requests
from datetime import datetime

# Define base paths
DATA_DIR = "data"
SCRAPED_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs")
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")

# Target subdirectories registry
CATEGORIES = [
    "constitution",
    "all_legislations",
    "case_laws",
    "treaties",
    "eac_legislations",
    "bills",
    "county_legislations",
    "kenya_gazette"
]

def init_environment():
    """Creates the data directories and an empty manifest if they don't exist."""
    os.makedirs(DATA_DIR, exist_ok=True)
    for category in CATEGORIES:
        os.makedirs(os.path.join(SCRAPED_DIR, category), exist_ok=True)
        
    if not os.path.exists(MANIFEST_PATH):
        initial_manifest = {
            "system_metadata": {
                "last_updated": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
                "total_indexed": 0,
                "total_downloaded": 0
            },
            "registry": {}
        }
        with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
            json.dump(initial_manifest, f, indent=2)
        print("-> Initialized fresh data directory and manifest tracking file.")
    else:
        print("-> Verified existing manifest and data directories.")

def load_manifest():
    """Safely loads manifest records from disk using direct string parsing to avoid file pointer collision."""
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        # Handle blank templates cleanly
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
            
        try:
            # FIX: Parse the read string directly instead of running json.load(f) on a drained file pointer
            return json.loads(content)
        except json.JSONDecodeError:
            print("[WARN] Manifest corrupted or malformed. Re-initializing empty system database structure.")
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}


def save_manifest(manifest_data):
    """Writes updated state back to disk atomically."""
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

# Run environment setup
init_environment()

                The consitution of Kenya

                        1. index the file and download it

In [ ]:
def download_constitution():
    """Registers and executes a direct download of the Constitution of Kenya 2010."""
    manifest = load_manifest()
    
    # Unique system key for the tracking item
    doc_key = "ke_constitution_2010"
    
    # Destination setup
    local_rel_path = os.path.join("data", "scraped_raw_pdfs", "constitution", "kenya_constitution_2010.pdf")
    local_abs_path = os.path.join(SCRAPED_DIR, "constitution", "kenya_constitution_2010.pdf")
    
    # Define source target (Official Parliament/Kenya Law mirror layout)
    source_url = "https://www.parliament.go.ke/sites/default/files/2017-05/The_Constitution_of_Kenya_2010.pdf"
    
    # Idempotency Guard check
    if doc_key in manifest["registry"] and manifest["registry"][doc_key]["downloaded"]:
        if os.path.exists(local_abs_path):
            print(f"-> [SKIP] {manifest['registry'][doc_key]['title']} already downloaded locally.")
            return

    print("-> Registering Constitution entry in database manifest...")
    manifest["registry"][doc_key] = {
        "category": "constitution",
        "title": "Constitution of Kenya 2010",
        "source_url": source_url,
        "local_path": local_rel_path,
        "downloaded": False,
        "downloaded_at": None
    }
    save_manifest(manifest)
    
    # Execute network stream request
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    print(f"-> Downloading resource from: {source_url}")
    try:
        response = requests.get(source_url, headers=headers, stream=True, timeout=30)
        response.raise_for_status()
        
        # Write binary stream safely chunk by chunk
        with open(local_abs_path, "wb") as pdf_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    pdf_file.write(chunk)
                    
        # Update successful registration status
        manifest = load_manifest()
        manifest["registry"][doc_key]["downloaded"] = True
        manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
        save_manifest(manifest)
        print("-> [SUCCESS] Constitution downloaded and manifest record checked off!")
        
    except Exception as e:
        print(f"-> [ERROR] Failed to fetch or write the constitution file: {str(e)}")

# Execute single-file extraction pipeline
download_constitution()

                All Legislations

                        1. Index the files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin

# ==========================================
# GLOBAL SCALING FACTOR (CONTROL PANEL)
# ==========================================
LIMIT_PAGES = 2     # Maximum number of pagination pages to crawl
LIMIT_PER_PAGE = 5  # Maximum files to index per page

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "all_legislations"

# FIX: Explicitly define the target download folder path structure
TARGET_DOWNLOAD_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs", CATEGORY)

def load_manifest():
    """Ensures the tracking json file exists and reads its current state."""
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {
            "system_metadata": {
                "last_updated": "",
                "total_indexed": 0,
                "total_downloaded": 0
            },
            "registry": {}
        }
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        return json.loads(content)

def save_manifest(manifest_data):
    """Saves updated registry items back to the JSON manifest document."""
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def index_all_legislations():
    # Utilizing the structural domain pointing to the active repository layout
    base_url = "https://new.kenyalaw.org/legislation/all"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # FIX: Force the creation of 'data/scraped_raw_pdfs/all_legislations' folder chain immediately
    print(f"--> Creating target folder structure: {TARGET_DOWNLOAD_DIR}")
    os.makedirs(TARGET_DOWNLOAD_DIR, exist_ok=True)
    
    manifest = load_manifest()
    current_url = base_url
    page_counter = 0
    total_indexed_this_run = 0
    
    print(f"--> Initializing Indexer Engine for: [{CATEGORY}]")
    print(f"--> Mode: {'SAMPLE EXPERIMENT' if LIMIT_PAGES else 'FULL PRODUCTION'}")
    
    while current_url:
        if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
            print(f"\n[INFO] Reached maximum experimental page limit ({LIMIT_PAGES}). Stopping indexer.")
            break
            
        page_counter += 1
        print(f"\n--- Processing Table Page {page_counter} ---")
        print(f"Target URL: {current_url}")
        
        try:
            response = requests.get(current_url, headers=headers, timeout=20)
            response.raise_for_status()
        except Exception as e:
            print(f"[ERROR] Connection failure on index page {page_counter}: {e}")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Isolate rows containing the target cell structure
        table_rows = soup.find_all("tr")
        if not table_rows:
            print("[WARN] No layout data table rows found on this page. Exiting.")
            break
            
        items_indexed_on_this_page = 0
        
        for row in table_rows:
            if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                print(f" -> Quota of {LIMIT_PER_PAGE} files reached for this page. Skipping remaining rows.")
                break
            
            # STRICT FILTER: Find the exact td cell styled with class="cell-title"
            title_td = row.find("td", class_="cell-title")
            if not title_td:
                continue  
                
            # Extract target landing page link
            link = title_td.find("a", href=True)
            if not link:
                continue
                
            view_url = urljoin(base_url, link["href"])
            title_text = link.get_text(strip=True)
            
            # Clean lookahead parameters to build an isolated track ID key
            url_clean = view_url.strip("/")
            segments = url_clean.split("/")
            internal_id = segments[-1] if segments[-1].replace("-", "").isalnum() else "doc_" + str(hash(view_url) % 100000)
            doc_key = f"ke_leg_id_{internal_id}"
            
            if doc_key in manifest["registry"]:
                print(f" [ALREADY INDEXED] Skipping item: {doc_key}")
                continue
                
            print(f" Found target cell: '{title_text[:50]}...'")
            download_link = None
            
            if "/source" in view_url.lower() or ".pdf" in view_url.lower():
                download_link = view_url
                print(f"   -> Direct asset link discovered within table layout.")
            else:
                print(f"   -> Loading subpage layout: {view_url}")
                try:
                    time.sleep(1.0)  # Politeness boundary delay
                    sub_response = requests.get(view_url, headers=headers, timeout=15)
                    sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                    
                    download_btn = sub_soup.find("a", class_=lambda c: c and "btn-primary" in c and "btn-shrink-sm" in c)
                    
                    if download_btn and download_btn.get("href"):
                        download_link = urljoin(view_url, download_btn["href"])
                        print(f"   -> Successfully extracted strict direct download path.")
                    else:
                        for sub_link in sub_soup.find_all("a", href=True):
                            if "/source" in sub_link["href"].lower():
                                download_link = urljoin(view_url, sub_link["href"])
                                break
                                
                    if not download_link:
                        download_link = view_url  
                except Exception as sub_err:
                    print(f" [ERROR] Failed to extract sub-page metadata: {sub_err}")
                    continue
            
            clean_filename = f"legislation_act_{internal_id}.pdf"
            
            # FIX: Explicitly maps the string reference into the clean folder layout
            local_rel_path = os.path.join(TARGET_DOWNLOAD_DIR, clean_filename)
            
            manifest["registry"][doc_key] = {
                "category": CATEGORY,
                "title": title_text,
                "source_url": download_link,
                "local_path": local_rel_path,
                "downloaded": False,
                "downloaded_at": None
            }
            
            items_indexed_on_this_page += 1
            total_indexed_this_run += 1
            
        save_manifest(manifest)
        
        # --- PAGINATION TRAVERSAL ---
        next_page_url = None
        pagination_block = soup.find("ul", class_="pagination") or soup.find("nav") or soup
        next_link = pagination_block.find("a", rel="next") or \
                    pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
                    
        if next_link and next_link.get("href"):
            next_page_url = urljoin(base_url, next_link["href"])
            
        current_url = next_page_url
        if not current_url:
            print("\n[INFO] No further page elements detected. End of directory trail.")
            break
            
    print(f"\n--> Indexing execution completed. Successfully registered {total_indexed_this_run} entries.")

index_all_legislations()


                        2. Downloading all the legislatioons indexed files

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")

def load_manifest():
    """Reads the current state from disk safely."""
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at: {MANIFEST_PATH}")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    """Saves updated registry items back to the JSON manifest document."""
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def run_downloader_engine():
    manifest = load_manifest()
    if not manifest or "registry" not in manifest:
        print("[ERROR] No valid registry found to download from.")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Gather items that haven't been downloaded yet
    pending_items = {k: v for k, v in manifest["registry"].items() if not v.get("downloaded")}
    
    if not pending_items:
        print("-> [INFO] All items are already downloaded and checked off!")
        return

    total_items = len(pending_items)
    print(f"--> Found {total_items} pending items to download.")

    for index, (doc_key, item) in enumerate(pending_items.items(), start=1):
        title = item.get("title", "Unknown Title")
        source_url = item.get("source_url")
        
        # FIX: Directly read the exact 'local_path' saved in the manifest by the indexer
        target_path = item.get("local_path")
        
        if not source_url or not target_path:
            print(f" [{index}/{total_items}] Skipping {title}: Missing URL or local path mapping.")
            continue

        print(f"\n[{index}/{total_items}] Processing: {title}...")
        print(f"    Source: {source_url}")
        print(f"    Target: {target_path}")

        # FIX: Auto-ensure parent directory branches (data/scraped_raw_pdfs/all_legislations) exist
        parent_dir = os.path.dirname(target_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)

        try:
            time.sleep(1.0) # Polite rate limiting delay
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()

            # Stream binary file contents chunk by chunk
            with open(target_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)

            # Update successful status tracking markers
            item["downloaded"] = True
            item["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            
            # Save progress incrementally after each asset is grabbed
            save_manifest(manifest)
            print(f"    [SUCCESS] Finished and registered entry to manifest.")

        except Exception as e:
            print(f"    [ERROR] Failed network or write block: {str(e)}")
            # Cleanup broken/partial downloads if they were created
            if os.path.exists(target_path):
                os.remove(target_path)

    # Re-calculate final system metadata tallies
    manifest["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest["registry"].values() if item.get("downloaded")
    )
    save_manifest(manifest)
    print("\n--> Download batch session completed.")

# Execute downloader pipeline
run_downloader_engine()


                All Case laws

                        1. Indexing files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin

# =====================================================================
# GLOBAL SCALING FACTOR (CONTROL PANEL FOR CASE LAWS)
# =====================================================================
# SAMPLE CONFIGURATION
YEARS_TO_CRAWL = [2024]  # Target only 2024 for the experiment
LIMIT_PAGES = 2         # Maximum number of pagination pages per year
LIMIT_PER_PAGE = 5      # Maximum files to index per page

# PRODUCTION SCALING CONFIGURATION (Uncomment below when ready to scale)
# YEARS_TO_CRAWL = list(range(1930, 2027)) # All years from 1930 to 2026
# LIMIT_PAGES = None                       # Crawl every single page
# LIMIT_PER_PAGE = None                    # Crawl every single row

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "case_laws"

# FIX: Establish the strict folder layout string mapping
TARGET_DOWNLOAD_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs", CATEGORY)

def load_manifest():
    """Reads the current state from disk safely, handling blank formats."""
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        return json.loads(content)

def save_manifest(manifest_data):
    """Saves updated registry items back to the JSON manifest document."""
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def index_case_laws():
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # FIX: Explicitly enforce the structural creation of target folders right away
    print(f"--> Creating nested path structure: {TARGET_DOWNLOAD_DIR}")
    os.makedirs(TARGET_DOWNLOAD_DIR, exist_ok=True)
    
    manifest = load_manifest()
    total_indexed_this_run = 0
    
    print(f"--> Initializing Indexer Engine for Category: [{CATEGORY}]")
    print(f"--> Target Years: {YEARS_TO_CRAWL}")
    
    # 1. LOOP THROUGH THE YEARS LAYER
    for year in YEARS_TO_CRAWL:
        # Utilizing the active digital tracking system URL mapping
        base_year_url = f"https://new.kenyalaw.org/judgments/all/{year}/"
        current_url = base_year_url
        page_counter = 0
        
        print(f"\n=========================================")
        print(f"   COMMENCING CRAWL FOR YEAR: {year}     ")
        print(f"=========================================")
        
        # 2. LOOP THROUGH THE PAGINATION LAYER
        while current_url:
            if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
                print(f"\n[INFO] Reached page limit ({LIMIT_PAGES}) for year {year}. Advancing year.")
                break
                
            page_counter += 1
            print(f"\n--- [Year {year}] Processing Page {page_counter} ---")
            print(f"URL: {current_url}")
            
            try:
                response = requests.get(current_url, headers=headers, timeout=20)
                response.raise_for_status()
            except Exception as e:
                print(f"[ERROR] Failed to fetch page {page_counter} for year {year}: {e}")
                break
                
            soup = BeautifulSoup(response.text, "html.parser")
            
            # Locate rows within the judgment directory table layout
            table_rows = soup.find_all("tr")
            if not table_rows:
                print(f"[WARN] No rows discovered on page {page_counter}. Stopping year {year}.")
                break
                
            items_indexed_on_this_page = 0
            
            # 3. LOOP THROUGH THE DOCUMENT ROW LAYER
            for row in table_rows:
                # FIX: Check limit based on total rows encountered on this page 
                if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                    print(f" -> Quota limit of {LIMIT_PER_PAGE} rows checked for this page. Moving to next page.")
                    break
                
                # FIX: Strict parsing using td cells with class="cell-title" to bypass headers and side navs
                title_td = row.find("td", class_="cell-title")
                if not title_td:
                    continue  # Safely bypass row fragments that don't match criteria
                
                link = title_td.find("a", href=True)
                if not link:
                    continue
                
                href = link["href"]
                view_url = urljoin(base_year_url, href)
                title_text = link.get_text(strip=True)
                
                # Keep track of elements evaluated against your control panel limit
                items_indexed_on_this_page += 1
                
                # Extract internal database id safely from Akoma Ntoso slugs or standard URL structures
                url_clean = view_url.strip("/")
                segments = url_clean.split("/")
                internal_id = segments[-1] if segments[-1].replace("-", "").isalnum() else "case_" + str(hash(view_url) % 100000)
                doc_key = f"ke_case_id_{internal_id}"
                
                # Idempotency Guard check
                if doc_key in manifest["registry"]:
                    print(f"   [ALREADY INDEXED] Skipping duplicate: {doc_key}")
                    continue
                    
                print(f"   Found Case: '{title_text[:50]}...'")
                print(f"   Stepping into subpage: {view_url}")
                
                # Step directly into the inner case page to extract the download source
                try:
                    time.sleep(1.0)  # Polite spacing delay
                    sub_response = requests.get(view_url, headers=headers, timeout=15)
                    sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                    
                    # Search inner page layout for direct download button paths
                    download_link = None
                    download_btn = sub_soup.find("a", class_=lambda c: c and "btn-primary" in c and "btn-shrink-sm" in c)
                    
                    if download_btn and download_btn.get("href"):
                        download_link = urljoin(view_url, download_btn["href"])
                    else:
                        # Fallback filtering strategy matching keywords
                        for sub_link in sub_soup.find_all("a", href=True):
                            href_lower = sub_link["href"].lower()
                            if "download" in href_lower or ".pdf" in href_lower or "/source" in href_lower:
                                download_link = urljoin(view_url, sub_link["href"])
                                break
                                
                    if not download_link:
                        download_link = view_url
                        
                    # Target path naming schema matching directory map configurations
                    clean_filename = f"case_id_{internal_id}.pdf"
                    
                    # FIX: Explicitly map local_rel_path using TARGET_DOWNLOAD_DIR to secure formatting
                    local_rel_path = os.path.join(TARGET_DOWNLOAD_DIR, clean_filename)
                    
                    # Update manifest dictionary snapshot
                    manifest["registry"][doc_key] = {
                        "category": CATEGORY,
                        "title": title_text,
                        "source_url": download_link,
                        "local_path": local_rel_path,
                        "downloaded": False,
                        "downloaded_at": None
                    }
                    
                    total_indexed_this_run += 1
                    
                except Exception as sub_err:
                    print(f"   [ERROR] Failed to extract subpage metadata: {sub_err}")
                    continue
                    
            # Commit data chunk instantly to file at the end of each page block
            save_manifest(manifest)
            
            # --- PAGINATION TRAVERSAL FOR CURRENT YEAR ---
            next_page_url = None
            pagination_block = soup.find("ul", class_="pagination") or soup.find("nav") or soup
            
            next_link = pagination_block.find("a", rel="next") or \
                        pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
                        
            if next_link and next_link.get("href"):
                next_page_url = urljoin(base_year_url, next_link["href"])
            else:
                for a_tag in soup.find_all("a", href=True):
                    if f"page={page_counter + 1}" in a_tag["href"]:
                        next_page_url = urljoin(base_year_url, a_tag["href"])
                        break
                        
            current_url = next_page_url
            if not current_url:
                print(f"\n[INFO] Completed all available pages for year {year}.")
                break
            print(f"\n--> Indexing complete. Registered {total_indexed_this_run} case law entries to data_manifest.json.")

index_case_laws()

                        2. Downloading all case laws files indexed 

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
TARGET_CATEGORY = "case_laws"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at {MANIFEST_PATH}. Run the Case Law indexer first!")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def download_indexed_case_laws():
    manifest = load_manifest()
    if not manifest:
        return

    # Filter for un-downloaded case law entries
    download_queue = [
        (doc_key, item) for doc_key, item in manifest["registry"].items()
        if item["category"] == TARGET_CATEGORY and not item["downloaded"]
    ]
    
    total_queue_count = len(download_queue)
    print(f"--> Found {total_queue_count} pending files to download for category: [{TARGET_CATEGORY}]")
    
    if total_queue_count == 0:
        print("--> No pending files to download. Your Case Law manifest entries are fully synced!")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    success_count = 0
    
    for index, (doc_key, item) in enumerate(download_queue, start=1):
        title = item["title"]
        source_url = item["source_url"]
        
        # FIX: Pull the literal path directly from your manifest records
        local_dest_path = item["local_path"]
        
        # Auto-create case_laws folder path safely if missing on disk
        parent_dir = os.path.dirname(local_dest_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        
        print(f"\n[{index}/{total_queue_count}] Processing: {title[:50]}...")
        print(f"    Source Link: {source_url}")
        print(f"    Target Path: {local_dest_path}")
        
        try:
            time.sleep(1.5)  # Polite sleep gap to manage server rate limits safely
            
            # Request payload stream
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            # Stream directly into local binary format block by block
            with open(local_dest_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)
            
            # --- MANIFEST STATE UPDATE ---
            # Reload current instance from disk to maintain updates from other runs
            manifest = load_manifest()
            manifest["registry"][doc_key]["downloaded"] = True
            manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            save_manifest(manifest)
            
            print(f"    [SUCCESS] File saved and marked complete in manifest.")
            success_count += 1
            
        except Exception as e:
            print(f"    [DOWNLOAD ERROR] Skipping entry due to connection failure: {e}")
            # Clean up broken or empty files if they get created on failure
            if os.path.exists(local_dest_path):
                os.remove(local_dest_path)
            continue

    print(f"\n--> Downloader run finished: Successfully pulled {success_count}/{total_queue_count} case law files.")

# Trigger downloader execution
download_indexed_case_laws()


                Treaties

                        1. Indexing files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin

# =====================================================================
# GLOBAL SCALING FACTOR (CONTROL PANEL FOR TREATIES)
# =====================================================================
# SAMPLE CONFIGURATION
LIMIT_PAGES = 2         # Maximum number of pagination pages to crawl
LIMIT_PER_PAGE = 5      # Maximum files to index per page

# PRODUCTION SCALING CONFIGURATION (Uncomment below when ready to scale)
# LIMIT_PAGES = None                       # Crawl every single page
# LIMIT_PER_PAGE = None                    # Crawl every single row

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "treaties"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def index_treaties():
    base_url = "https://kenyalaw.org/taxonomy/collections/collections-treaties"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    manifest = load_manifest()
    current_url = base_url
    page_counter = 0
    total_indexed_this_run = 0
    
    print(f"--> Initializing Indexer Engine for Category: [{CATEGORY}]")
    print(f"--> Mode: {'SAMPLE EXPERIMENT' if LIMIT_PAGES else 'FULL PRODUCTION'}")
    
    while current_url:
        if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
            print(f"\n[INFO] Reached maximum experimental page limit ({LIMIT_PAGES}). Stopping indexer.")
            break
            
        page_counter += 1
        print(f"\n--- Processing Treaties Page {page_counter} ---")
        print(f"Target URL: {current_url}")
        
        try:
            response = requests.get(current_url, headers=headers, timeout=20)
            response.raise_for_status()
        except Exception as e:
            print(f"[ERROR] Connection failure on index page {page_counter}: {e}")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Check standard taxonomy structures (table rows or list blocks)
        table_rows = soup.find_all("tr") or soup.select(".views-row") or soup.find_all("li", class_="treaty-item")
        
        # Fallback to general table rows if structural selections are blank
        if not table_rows:
            table_rows = soup.find_all("tr")
            
        if not table_rows:
            print("[WARN] No collection data elements or rows found on this page. Exiting.")
            break
            
        items_indexed_on_this_page = 0
        
        for row in table_rows:
            if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                print(f" -> Quota of {LIMIT_PER_PAGE} files reached for this page. Skipping remaining rows.")
                break
                
            # Locate anchor links pointing to individual treaty profiles or view items
            links = row.find_all("a", href=True)
            view_url = None
            title_text = ""
            
            for link in links:
                # Check for standard paths or direct view routes common to taxonomies
                href = link["href"]
                if "/treaties/view/" in href or "/view/" in href or "id=" in href:
                    view_url = urljoin(base_url, href)
                    title_text = link.get_text(strip=True) or row.get_text(strip=True)
                    break
            
            # Universal fallback link discovery if custom taxonomy filters miss it
            if not view_url and links:
                for link in links:
                    if not any(x in link["href"].lower() for x in ["page=", "sort=", "javascript"]):
                        view_url = urljoin(base_url, link["href"])
                        title_text = link.get_text(strip=True)
                        break
                        
            if not view_url:
                continue
                
            # Skip common empty navigation table templates or sorting headers
            if not title_text or "title" in title_text.lower() or "date" in title_text.lower():
                continue
                
            # Generate a clean signature key based on the final fragment of the URL path or ID parameter
            internal_id = view_url.rstrip("/").split("/")[-1].split("=")[-1]
            doc_key = f"ke_treaty_id_{internal_id}"
            
            # Idempotency Guard
            if doc_key in manifest["registry"]:
                print(f"   [ALREADY INDEXED] Skipping item: {doc_key}")
                continue
                
            print(f"   Found Treaty: '{title_text[:50]}...'")
            print(f"   Stepping into detail sub-page: {view_url}")
            
            # Hit inner page structure to look for download asset hooks
            try:
                time.sleep(1.0)  # Rate limiting delay
                sub_response = requests.get(view_url, headers=headers, timeout=15)
                sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                
                download_link = None
                for sub_link in sub_soup.find_all("a", href=True):
                    href_lower = sub_link["href"].lower()
                    if "download" in href_lower or ".pdf" in href_lower or "/source" in href_lower:
                        download_link = urljoin(view_url, sub_link["href"])
                        break
                
                if not download_link:
                    download_link = view_url
                    
                # Normalize filename targeting the 'treaties' category subdirectory
                clean_filename = f"treaty_{internal_id}.pdf"
                local_rel_path = os.path.join("data", "scraped_raw_pdfs", CATEGORY, clean_filename)
                
                # Append item to registry
                manifest["registry"][doc_key] = {
                    "category": CATEGORY,
                    "title": title_text,
                    "source_url": download_link,
                    "local_path": local_rel_path,
                    "downloaded": False,
                    "downloaded_at": None
                }
                
                items_indexed_on_this_page += 1
                total_indexed_this_run += 1
                
            except Exception as sub_err:
                print(f"   [ERROR] Failed to index sub-page metadata: {sub_err}")
                continue
                
        # Save tracking data updates to manifest immediately
        save_manifest(manifest)
        
        # --- PAGINATION TRAVERSAL ---
        next_page_url = None
        pagination_block = soup.find("ul", class_="pagination") or soup.find(".pagination-next") or soup
        
        next_link = pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
        if next_link and next_link.get("href"):
            next_page_url = urljoin(base_url, next_link["href"])
        else:
            # Fallback block searching via query parameters or active page step references
            for a_tag in soup.find_all("a", href=True):
                if f"page={page_counter}" in a_tag["href"]: # zero-indexed paginations check
                    next_page_url = urljoin(base_url, a_tag["href"])
                    break
            
        current_url = next_page_url
        if not current_url:
            print("\n[INFO] No further page buttons detected. End of treaties directory.")
            break
            
    print(f"\n--> Indexing execution completed. Successfully registered {total_indexed_this_run} documents to data_manifest.json.")

# Trigger the treaties indexer cell
index_treaties()

                        2. Download all the treaties indexed files

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
TARGET_CATEGORY = "treaties"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at {MANIFEST_PATH}. Run the Treaties indexer first!")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def download_indexed_treaties():
    manifest = load_manifest()
    if not manifest:
        return

    # Filter for un-downloaded treaty entries
    download_queue = [
        (doc_key, item) for doc_key, item in manifest["registry"].items()
        if item["category"] == TARGET_CATEGORY and not item["downloaded"]
    ]
    
    total_queue_count = len(download_queue)
    print(f"--> Found {total_queue_count} pending files to download for category: [{TARGET_CATEGORY}]")
    
    if total_queue_count == 0:
        print("--> No pending files to download. Your Treaties manifest entries are fully synced!")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    success_count = 0
    
    for index, (doc_key, item) in enumerate(download_queue, start=1):
        title = item["title"]
        source_url = item["source_url"]
        
        # FIX: Rely completely on the exact path value recorded by your indexer configuration
        local_dest_path = item["local_path"]
        
        # Auto-create treaties folder layout if missing safely on disk
        parent_dir = os.path.dirname(local_dest_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        
        print(f"\n[{index}/{total_queue_count}] Processing: {title[:50]}...")
        print(f"    Source Link: {source_url}")
        print(f"    Target Path: {local_dest_path}")
        
        try:
            time.sleep(1.5)  # Polite sleep gap to manage server rate limits safely
            
            # Request payload stream
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            # Stream directly into local binary format block by block
            with open(local_dest_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)
            
            # --- MANIFEST STATE UPDATE ---
            # Reload current instance from disk to maintain updates from other runs safely
            manifest = load_manifest()
            manifest["registry"][doc_key]["downloaded"] = True
            manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            save_manifest(manifest)
            
            print(f"    [SUCCESS] File saved and marked complete in manifest.")
            success_count += 1
            
        except Exception as e:
            print(f"    [DOWNLOAD ERROR] Skipping entry due to connection failure: {e}")
            # Structural Cleanup: Delete corrupted or 0-byte structural artifacts on write failures
            if os.path.exists(local_dest_path):
                os.remove(local_dest_path)
            continue

    print(f"\n--> Downloader run finished: Successfully pulled {success_count}/{total_queue_count} treaty files.")

# Trigger downloader execution
download_indexed_treaties()


                EAC Legislations

                        1. Indexing the files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin

# =====================================================================
# GLOBAL SCALING FACTOR (CONTROL PANEL FOR EAC LEGISLATIONS)
# =====================================================================
# SAMPLE CONFIGURATION
LIMIT_PAGES = 2         # Maximum number of pagination pages to crawl
LIMIT_PER_PAGE = 5      # Maximum files to index per page

# PRODUCTION SCALING CONFIGURATION (Uncomment below when ready to scale)
# LIMIT_PAGES = None                       # Crawl every single page
# LIMIT_PER_PAGE = None                    # Crawl every single row

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "eac_legislations"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def index_eac_legislations():
    base_url = "https://kenyalaw.org/taxonomy/foreign-legislation/foreign-legislation-east-african-community-eac"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    manifest = load_manifest()
    current_url = base_url
    page_counter = 0
    total_indexed_this_run = 0
    
    print(f"--> Initializing Indexer Engine for Category: [{CATEGORY}]")
    print(f"--> Mode: {'SAMPLE EXPERIMENT' if LIMIT_PAGES else 'FULL PRODUCTION'}")
    
    while current_url:
        if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
            print(f"\n[INFO] Reached maximum experimental page limit ({LIMIT_PAGES}). Stopping indexer.")
            break
            
        page_counter += 1
        print(f"\n--- Processing EAC Legislations Page {page_counter} ---")
        print(f"Target URL: {current_url}")
        
        try:
            response = requests.get(current_url, headers=headers, timeout=20)
            response.raise_for_status()
        except Exception as e:
            print(f"[ERROR] Connection failure on index page {page_counter}: {e}")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Target standard taxonomy row containers
        table_rows = soup.find_all("tr") or soup.select(".views-row") or soup.find_all("li")
        
        if not table_rows:
            print("[WARN] No layout data rows found on this page. Exiting.")
            break
            
        items_indexed_on_this_page = 0
        
        for row in table_rows:
            if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                print(f" -> Quota of {LIMIT_PER_PAGE} files reached for this page. Skipping remaining rows.")
                break
                
            # Scan for item view profile links
            links = row.find_all("a", href=True)
            view_url = None
            title_text = ""
            
            for link in links:
                href = link["href"]
                # Capture paths containing standard view nodes or parameters
                if "/view/" in href or "id=" in href or "foreign-legislation" in href:
                    if not any(x in href.lower() for x in ["page=", "sort="]):
                        view_url = urljoin(base_url, href)
                        title_text = link.get_text(strip=True) or row.get_text(strip=True)
                        break
                        
            if not view_url and links:
                # Blind fallback mapping for plain list views
                for link in links:
                    if not any(x in link["href"].lower() for x in ["page=", "sort=", "javascript"]):
                        view_url = urljoin(base_url, link["href"])
                        title_text = link.get_text(strip=True)
                        break
                        
            if not view_url:
                continue
                
            if not title_text or "title" in title_text.lower() or "date" in title_text.lower():
                continue
                
            # Cleanly isolate the unique ID signature from the terminal fragment of the path
            internal_id = view_url.rstrip("/").split("/")[-1].split("=")[-1]
            doc_key = f"ke_eac_id_{internal_id}"
            
            # Idempotency Guard
            if doc_key in manifest["registry"]:
                print(f"   [ALREADY INDEXED] Skipping item: {doc_key}")
                continue
                
            print(f"   Found EAC Doc: '{title_text[:50]}...'")
            print(f"   Stepping into detail sub-page: {view_url}")
            
            # Hit detail page to extract raw source file
            try:
                time.sleep(1.0)  # Standard politeness delay
                sub_response = requests.get(view_url, headers=headers, timeout=15)
                sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                
                download_link = None
                for sub_link in sub_soup.find_all("a", href=True):
                    href_lower = sub_link["href"].lower()
                    if "download" in href_lower or ".pdf" in href_lower or "/source" in href_lower:
                        download_link = urljoin(view_url, sub_link["href"])
                        break
                        
                if not download_link:
                    download_link = view_url
                    
                # Standardize relative paths and filenames under the 'eac_legislations' folder
                clean_filename = f"eac_{internal_id}.pdf"
                local_rel_path = os.path.join("data", "scraped_raw_pdfs", CATEGORY, clean_filename)
                
                # Register metadata object block
                manifest["registry"][doc_key] = {
                    "category": CATEGORY,
                    "title": title_text,
                    "source_url": download_link,
                    "local_path": local_rel_path,
                    "downloaded": False,
                    "downloaded_at": None
                }
                
                items_indexed_on_this_page += 1
                total_indexed_this_run += 1
                
            except Exception as sub_err:
                print(f"   [ERROR] Failed to extract sub-page metadata: {sub_err}")
                continue
                
        # Commit row chunk data checkpoints instantly
        save_manifest(manifest)
        
        # --- PAGINATION TRAVERSAL ---
        next_page_url = None
        pagination_block = soup.find("ul", class_="pagination") or soup.find(".pagination-next") or soup
        
        next_link = pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
        if next_link and next_link.get("href"):
            next_page_url = urljoin(base_url, next_link["href"])
        else:
            for a_tag in soup.find_all("a", href=True):
                if f"page={page_counter}" in a_tag["href"]:
                    next_page_url = urljoin(base_url, a_tag["href"])
                    break
                    
        current_url = next_page_url
        if not current_url:
            print("\n[INFO] End of available pages for EAC Legislations directory.")
            break
            
    print(f"\n--> Indexing execution completed. Successfully registered {total_indexed_this_run} documents to data_manifest.json.")

# Trigger indexer execution
index_eac_legislations()

                        2. Downloading all the EAC legislation indexed files

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
TARGET_CATEGORY = "eac_legislations"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at {MANIFEST_PATH}. Run the EAC indexer first!")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def download_indexed_eac_legislations():
    manifest = load_manifest()
    if not manifest:
        return

    # Filter out pending un-downloaded items matching our category target
    download_queue = [
        (doc_key, item) for doc_key, item in manifest["registry"].items()
        if item["category"] == TARGET_CATEGORY and not item["downloaded"]
    ]
    
    total_queue_count = len(download_queue)
    print(f"--> Found {total_queue_count} pending files to download for category: [{TARGET_CATEGORY}]")
    
    if total_queue_count == 0:
        print("--> No pending files to download. Your EAC registry entries are completely up to date!")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    success_count = 0
    
    for index, (doc_key, item) in enumerate(download_queue, start=1):
        title = item["title"]
        source_url = item["source_url"]
        
        # FIX: Directly use the literal path value already recorded in the manifest entries
        local_dest_path = item["local_path"]
        
        # Confirm subdirectory target exists safely on disk prior to writing stream
        parent_dir = os.path.dirname(local_dest_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        
        print(f"\n[{index}/{total_queue_count}] Processing: {title[:50]}...")
        print(f"    Source Link: {source_url}")
        print(f"    Target Path: {local_dest_path}")
        
        try:
            time.sleep(1.5)  # Defensive throttling pause
            
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            with open(local_dest_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)
            
            # --- MANIFEST STATE SYNC ---
            manifest = load_manifest()
            manifest["registry"][doc_key]["downloaded"] = True
            manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            save_manifest(manifest)
            
            print(f"    [SUCCESS] Document saved safely and flag set to True.")
            success_count += 1
            
        except Exception as e:
            print(f"    [DOWNLOAD ERROR] Connection interrupted or file missed: {e}")
            # Structural Guard: Drop empty or corrupted file fragments if data streaming cuts out
            if os.path.exists(local_dest_path):
                os.remove(local_dest_path)
            continue

    print(f"\n--> Downloader run finished: Successfully pulled {success_count}/{total_queue_count} EAC files.")

# Execute downloader function
download_indexed_eac_legislations()


                County Legislations

                        1. Indexing the files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin

# =====================================================================
# GLOBAL SCALING FACTOR (CONTROL PANEL FOR COUNTY LEGISLATIONS)
# =====================================================================
# SAMPLE CONFIGURATION
COUNTIES_TO_CRAWL = ["ke-001", "ke-030"]  # ke-001 (Mombasa), ke-030 (Baringo)
LIMIT_PAGES = 2                            # Maximum pagination pages per county
LIMIT_PER_PAGE = 5                         # Maximum files to index per page

# PRODUCTION SCALING CONFIGURATION (Uncomment below when ready to scale)
# COUNTIES_TO_CRAWL = "ALL"                # Auto-discover all 47 counties from the main link
# LIMIT_PAGES = None                       # Crawl every single page
# LIMIT_PER_PAGE = None                    # Crawl every single row

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "county_legislations"

# FIX: Define the explicit nested system directory architecture 
TARGET_DOWNLOAD_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs", CATEGORY)

def load_manifest():
    """Safely loads manifest records from disk using direct string parsing to avoid file pointer collision."""
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        # Handle blank templates cleanly
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
            
        try:
            # FIX: Parse the read string directly instead of running json.load(f) on a drained file pointer
            return json.loads(content)
        except json.JSONDecodeError:
            print("[WARN] Manifest corrupted or malformed. Re-initializing empty system database structure.")
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}


def save_manifest(manifest_data):
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def discover_all_counties(main_url, headers):
    """Parses the main directory to harvest clean endpoint paths for all 47 counties."""
    print("[*] Scaling mode active. Scanning main page to discover all 47 counties...")
    try:
        response = requests.get(main_url, headers=headers, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        county_codes = []
        for link in soup.find_all("a", href=True):
            href = link["href"].rstrip("/")
            # FIX: Robust check matching your targeted snippet style <li><a href="/legislation/ke-030/">
            if "/legislation/ke-" in href or "/legislation/ke" in href:
                code = href.split("/legislation/")[-1].replace("/", "")
                if code and code not in county_codes:
                    county_codes.append(code)
        
        print(f"[✓] Successfully discovered {len(county_codes)} counties dynamically.")
        return county_codes
    except Exception as e:
        print(f"[ERROR] Failed to discover counties from main directory: {e}")
        return ["ke-030"] # Hard fallback to Baringo to avoid total block failure

def index_county_legislations():
    main_directory_url = "https://kenyalaw.org"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # FIX: Build the storage folders immediately on initialization
    print(f"--> Instantiating target layout folder tree: {TARGET_DOWNLOAD_DIR}")
    os.makedirs(TARGET_DOWNLOAD_DIR, exist_ok=True)
    
    manifest = load_manifest()
    total_indexed_this_run = 0
    
    # Resolve running mode (Sample vs Scaled Production)
    if COUNTIES_TO_CRAWL == "ALL":
        target_counties = discover_all_counties(main_directory_url, headers)
    else:
        target_counties = COUNTIES_TO_CRAWL
        
    print(f"--> Initializing Indexer Engine for Category: [{CATEGORY}]")
    print(f"--> Targets: {len(target_counties)} counties queued for execution.")
    
    # 1. LOOP THROUGH COUNTIES LAYER
    for county_code in target_counties:
        # FIX: Append /all to match the legitimate functional URL structure layout
        base_county_url = f"https://kenyalaw.org/legislation/{county_code}/all"
        current_url = base_county_url
        page_counter = 0
        
        print(f"\n=========================================")
        print(f"   PROCESSING COUNTY REGISTRY: {county_code.upper()} ")
        print(f"=========================================")
        
        # 2. LOOP THROUGH PAGINATION LAYER
        while current_url:
            if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
                print(f"[INFO] Reached page quota limit ({LIMIT_PAGES}) for {county_code}. Advancing county.")
                break
                
            page_counter += 1
            print(f"\n--- [{county_code.upper()}] Reading Page {page_counter} ---")
            print(f"URL: {current_url}")
            
            try:
                response = requests.get(current_url, headers=headers, timeout=20)
                response.raise_for_status()
            except Exception as e:
                print(f"[ERROR] Connection failed on page {page_counter} for county {county_code}: {e}")
                break
                
            soup = BeautifulSoup(response.text, "html.parser")
            table_rows = soup.find_all("tr")
            
            if not table_rows:
                print(f"[WARN] No records found on page {page_counter} for county {county_code}. Stopping tree traversal.")
                break
                
            items_indexed_on_this_page = 0
            
            # 3. LOOP THROUGH DOCUMENT ROW LAYER
            for row in table_rows:
                if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                    print(f" -> Quota of {LIMIT_PER_PAGE} files reached for this page. Shifting to next layout block.")
                    break
                
                # FIX: Strict filter matching your layout requirement to isolate target cell columns
                title_td = row.find("td", class_="cell-title")
                if not title_td:
                    continue  # Safely bypass headers or alternative navigation pieces
                    
                link = title_td.find("a", href=True)
                if not link:
                    continue
                    
                href = link["href"]
                # FIX: Broad match strategy allowing Akoma Ntoso schema pathways to pass safely
                if "/act/" in href or "/legislation/" in href or "ke-" in href:
                    view_url = urljoin(base_county_url, href)
                    title_text = link.get_text(strip=True)
                else:
                    continue
                    
                if not title_text or any(x in title_text.lower() for x in ["title", "date", "action"]):
                    continue
                
                # FIX: Safe and resilient internal unique key calculation sequence from Akoma structures
                url_clean = view_url.strip("/")
                segments = url_clean.split("/")
                
                # Safely find a combination of path segments to create a clean numeric or alphanumeric ID
                if len(segments) >= 3:
                    internal_id = f"{segments[-3]}_{segments[-2]}"
                else:
                    internal_id = str(hash(view_url) % 100000)
                    
                doc_key = f"ke_county_doc_{county_code}_{internal_id}"
                
                # Idempotency Guard
                if doc_key in manifest["registry"]:
                    print(f"   [ALREADY INDEXED] Skipping item: {doc_key}")
                    continue
                    
                print(f"   Found Legislation: '{title_text[:50]}...'")
                print(f"   Stepping into subpage: {view_url}")
                
                # Step inside detail layout to gather file hooks
                try:
                    time.sleep(1.0)  # Politeness factor
                    sub_response = requests.get(view_url, headers=headers, timeout=15)
                    sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                    
                    download_link = None
                    # FIX: Explicit selector seeking the exact bootstrap button layout specified
                    download_btn = sub_soup.find("a", class_=lambda c: c and "btn-primary" in c and "btn-shrink-sm" in c)
                    
                    if download_btn and download_btn.get("href"):
                        download_link = urljoin(view_url, download_btn["href"])
                    else:
                        # Fallback parsing strategy if style modifiers vary slightly on older items
                        for sub_link in sub_soup.find_all("a", href=True):
                            href_lower = sub_link["href"].lower()
                            if "download" in href_lower or ".pdf" in href_lower or "/source" in href_lower:
                                download_link = urljoin(view_url, sub_link["href"])
                                break
                            
                    if not download_link:
                        download_link = view_url
                        
                    # Structure clean path targets isolating by county folder maps
                    clean_filename = f"{county_code}_doc_{internal_id}.pdf"
                    
                    # FIX: Map local paths safely through the defined unified target download folder
                    local_rel_path = os.path.join(TARGET_DOWNLOAD_DIR, clean_filename)
                    
                    manifest["registry"][doc_key] = {
                        "category": CATEGORY,
                        "county_code": county_code,
                        "title": title_text,
                        "source_url": download_link,
                        "local_path": local_rel_path,
                        "downloaded": False,
                        "downloaded_at": None
                    }
                    
                    items_indexed_on_this_page += 1
                    total_indexed_this_run += 1
                    
                except Exception as sub_err:
                    print(f"   [ERROR] Failed to scrap subpage info: {sub_err}")
                    continue
                    
            # Instantly update manifest checkpoints 
            save_manifest(manifest)
            
            # --- PAGINATION TRAVERSAL WITHIN CURRENT COUNTY ---
            next_page_url = None
            pagination_block = soup.find("ul", class_="pagination") or soup.find("nav") or soup
            next_link = pagination_block.find("a", rel="next") or \
                        pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
            
            if next_link and next_link.get("href"):
                next_page_url = urljoin(base_county_url, next_link["href"])
            else:
                for a_tag in soup.find_all("a", href=True):
                    if f"page={page_counter + 1}" in a_tag["href"]:
                        next_page_url = urljoin(base_county_url, a_tag["href"])
                        break
                        
            current_url = next_page_url
            if not current_url:
                print(f"[INFO] Finished all pages for county {county_code}.")
                break
                
    print(f"\n--> Indexing complete. Registered {total_indexed_this_run} county legislation entries to data_manifest.json.")

# Run the county indexer cell
index_county_legislations()


                        2. Downloading all the County legislation indexed files

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
TARGET_CATEGORY = "county_legislations"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at {MANIFEST_PATH}. Run the County Legislation indexer first!")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def download_indexed_county_legislations():
    manifest = load_manifest()
    if not manifest:
        return

    # Filter registry items for un-downloaded county files
    download_queue = [
        (doc_key, item) for doc_key, item in manifest["registry"].items()
        if item["category"] == TARGET_CATEGORY and not item["downloaded"]
    ]
    
    total_queue_count = len(download_queue)
    print(f"--> Found {total_queue_count} pending files to download for category: [{TARGET_CATEGORY}]")
    
    if total_queue_count == 0:
        print("--> No pending files to download. Your County Legislation files are completely in sync!")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    success_count = 0
    
    for index, (doc_key, item) in enumerate(download_queue, start=1):
        title = item["title"]
        source_url = item["source_url"]
        
        # FIX: Directly use the literal path value already recorded in the manifest entries
        local_dest_path = item["local_path"]
        
        # Confirm subdirectory target exists safely on disk prior to writing stream
        parent_dir = os.path.dirname(local_dest_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        
        print(f"\n[{index}/{total_queue_count}] Processing: {title[:50]}...")
        print(f"    Source Link: {source_url}")
        print(f"    Target Path: {local_dest_path}")
        
        try:
            time.sleep(1.5)  # Defensive throttling pause
            
            # Request payload stream
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            with open(local_dest_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)
            
            # --- UPDATE MANIFEST STATE ---
            manifest = load_manifest()
            manifest["registry"][doc_key]["downloaded"] = True
            manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            save_manifest(manifest)
            
            print(f"    [SUCCESS] Saved file and flipped downloaded status flag.")
            success_count += 1
            
        except Exception as e:
            print(f"    [DOWNLOAD ERROR] Skipping target entry: {e}")
            # Structural Guard: Drop empty or corrupted file fragments if data streaming cuts out
            if os.path.exists(local_dest_path):
                os.remove(local_dest_path)
            continue

    print(f"\n--> Downloader run finished: Successfully pulled {success_count}/{total_queue_count} county files.")

# Execute downloader function
download_indexed_county_legislations()


                Bills

                        1. Indexing the files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin

# =====================================================================
# GLOBAL SCALING FACTOR (CONTROL PANEL FOR BILLS)
# =====================================================================
# SAMPLE CONFIGURATION
YEARS_TO_CRAWL = [2013, 2026]           # Specific years to target for sampling
LIMIT_PAGES = 2                        # Maximum pagination pages per year
LIMIT_PER_PAGE = 5                     # Maximum files to index per page

# PRODUCTION SCALING CONFIGURATION (Uncomment below when ready to scale)
# YEARS_TO_CRAWL = "ALL"                 # Automatically loops from 2013 to current year
# LIMIT_PAGES = None                      # Crawl every single page
# LIMIT_PER_PAGE = None                   # Crawl every single row

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "bills"

# FIX: Establish explicit folder target paths for file creation
TARGET_DOWNLOAD_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs", CATEGORY)

def load_manifest():
    """Safely loads manifest records using direct string parsing to avoid pointer errors."""
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
            
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            print("[WARN] Manifest corrupted or malformed. Re-initializing empty system database structure.")
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}

def save_manifest(manifest_data):
    manifest_data["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def resolve_target_years():
    """Generates years from 2013 to current year if ALL is specified."""
    if YEARS_TO_CRAWL == "ALL":
        current_year = datetime.now().year
        return list(range(2013, current_year + 1))
    return YEARS_TO_CRAWL

def index_bills():
    # FIX: Points directly to active current engine deployment mappings
    base_portal_url = "https://kenyalaw.org"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # FIX: Enforce physical creation of output folders immediately on start
    print(f"--> Instantiating target folder tree: {TARGET_DOWNLOAD_DIR}")
    os.makedirs(TARGET_DOWNLOAD_DIR, exist_ok=True)
    
    manifest = load_manifest()
    target_years = resolve_target_years()
    total_indexed_this_run = 0
    
    print(f"--> Initializing Indexer Engine for Category: [{CATEGORY}]")
    print(f"--> Targets: {len(target_years)} years queued for execution.")
    
    # 1. TIMELINE LAYER (YEAR STRATEGY)
    for year in target_years:
        # FIX: Swapped parameter token mapping structure from years= to year=
        base_year_url = f"https://kenyalaw.org/bills/?years={year}&q=&sort=-date"
        current_url = base_year_url
        page_counter = 0
        
        print(f"\n=========================================")
        print(f"   PROCESSING BILLS REGISTRY YEAR: {year} ")
        print(f"=========================================")
        
        # 2. PAGINATION LAYER
        while current_url:
            if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
                print(f"[INFO] Reached maximum page limit ({LIMIT_PAGES}) for year {year}. Advancing calendar.")
                break
                
            page_counter += 1
            print(f"\n--- [YEAR {year}] Reading Index Page {page_counter} ---")
            print(f"Target URL: {current_url}")
            
            try:
                response = requests.get(current_url, headers=headers, timeout=20)
                response.raise_for_status()
            except Exception as e:
                print(f"[ERROR] Connection failure on page {page_counter} for year {year}: {e}")
                break
                
            soup = BeautifulSoup(response.text, "html.parser")
            table_rows = soup.find_all("tr")
            
            if not table_rows:
                print(f"[WARN] No records found on page {page_counter} for year {year}. Halting pagination loop.")
                break
                
            items_indexed_on_this_page = 0
            
            # 3. DOCUMENT ROW LAYER
            for row in table_rows:
                if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                    print(f" -> Quota of {LIMIT_PER_PAGE} files reached for this page. Shifting window.")
                    break
                
                # FIX: Enforce strict parsing filtering using cell-title classes to isolate targeted link cells
                title_td = row.find("td", class_="cell-title")
                if not title_td:
                    continue  # Safely bypass non-compliant header fragments or metadata grids
                    
                link = title_td.find("a", href=True)
                if not link:
                    continue
                    
                href = link["href"]
                # FIX: Broadened route match parameter checks to let Akoma Ntoso /bill/ routing schemas pass
                if "/bill/" in href or "/view/" in href or "/legislation/" in href:
                    view_url = urljoin(base_year_url, href)
                    title_text = link.get_text(strip=True)
                else:
                    continue
                    
                if not title_text or any(x in title_text.lower() for x in ["title", "date", "action"]):
                    continue
                    
                # FIX: Safe extraction sequence splits cleanly across alphanumeric path nodes
                url_clean = view_url.strip("/")
                segments = url_clean.split("/")
                
                if len(segments) >= 2:
                    # Extracts unique tokens like year/number variations or structural hash keys
                    internal_id = f"{segments[-2]}_{segments[-1]}"
                else:
                    internal_id = str(hash(view_url) % 100000)
                    
                doc_key = f"ke_bill_{year}_{internal_id}"
                
                # Idempotency Guard
                if doc_key in manifest["registry"]:
                    print(f"   [ALREADY INDEXED] Skipping item: {doc_key}")
                    continue
                    
                print(f"   Found Bill Target: '{title_text[:50]}...'")
                print(f"   Stepping into detail view subpage: {view_url}")
                
                # Dig inside detail layer to parse target raw asset download link
                try:
                    time.sleep(1.0)  # Rate limiting courtesy delay
                    sub_response = requests.get(view_url, headers=headers, timeout=15)
                    sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                    
                    download_link = None
                    # FIX: Precision tracking hunting for the standard bootstrap file-delivery component buttons
                    download_btn = sub_soup.find("a", class_=lambda c: c and "btn-primary" in c and "btn-shrink-sm" in c)
                    
                    if download_btn and download_btn.get("href"):
                        download_link = urljoin(view_url, download_btn["href"])
                    else:
                        for sub_link in sub_soup.find_all("a", href=True):
                            href_lower = sub_link["href"].lower()
                            if "download" in href_lower or ".pdf" in href_lower or "/source" in href_lower:
                                download_link = urljoin(view_url, sub_link["href"])
                                break
                            
                    if not download_link:
                        download_link = view_url
                        
                    # Target relative file path schema
                    clean_filename = f"bill_{year}_{internal_id}.pdf"
                    
                    # FIX: Map local relative paths safely through the defined unified download directory string
                    local_rel_path = os.path.join(TARGET_DOWNLOAD_DIR, clean_filename)
                    
                    # Log object block mapping
                    manifest["registry"][doc_key] = {
                        "category": CATEGORY,
                        "year": year,
                        "title": title_text,
                        "source_url": download_link,
                        "local_path": local_rel_path,
                        "downloaded": False,
                        "downloaded_at": None
                    }
                    
                    items_indexed_on_this_page += 1
                    total_indexed_this_run += 1
                    
                except Exception as sub_err:
                    print(f"   [ERROR] Failed to extract sub-page info: {sub_err}")
                    continue
                    
            # Commit manifest snapshot immediately
            save_manifest(manifest)
            
            # --- PAGINATION TRAVERSAL WITHIN CURRENT YEAR ---
            next_page_url = None
            pagination_block = soup.find("ul", class_="pagination") or soup.find("nav") or soup
            next_link = pagination_block.find("a", rel="next") or \
                        pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
            
            if next_link and next_link.get("href"):
                next_page_url = urljoin(base_year_url, next_link["href"])
            else:
                for a_tag in soup.find_all("a", href=True):
                    # FIX: Query for lookahead lookups matching the upcoming sequential block parameter count (page_counter + 1)
                    if f"page={page_counter + 1}" in a_tag["href"] and f"year={year}" in a_tag["href"]:
                        next_page_url = urljoin(base_year_url, a_tag["href"])
                        break
                        
            current_url = next_page_url
            if not current_url:
                print(f"[INFO] Complete with pagination tree for year {year}.")
                break
                
    print(f"\n--> Indexing completed. Registered {total_indexed_this_run} bills to data_manifest.json.")

# Run the Bills indexer cell
index_bills()

                    


                        2. Downloading all the Bills indexed files

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
TARGET_CATEGORY = "bills"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at {MANIFEST_PATH}. Run the Bills indexer first!")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def download_indexed_bills():
    manifest = load_manifest()
    if not manifest:
        return

    # Filter for un-downloaded entries registered under bills
    download_queue = [
        (doc_key, item) for doc_key, item in manifest["registry"].items()
        if item["category"] == TARGET_CATEGORY and not item["downloaded"]
    ]
    
    total_queue_count = len(download_queue)
    print(f"--> Found {total_queue_count} pending files to download for category: [{TARGET_CATEGORY}]")
    
    if total_queue_count == 0:
        print("--> No pending files to download. Your Bills storage registry is completely updated!")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    success_count = 0
    
    for index, (doc_key, item) in enumerate(download_queue, start=1):
        title = item["title"]
        source_url = item["source_url"]
        
        # FIX: Directly use the literal path value already recorded in the manifest entries
        local_dest_path = item["local_path"]
        
        # Confirm subdirectory target exists safely on disk prior to writing stream
        parent_dir = os.path.dirname(local_dest_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        
        print(f"\n[{index}/{total_queue_count}] Processing: {title[:50]}...")
        print(f"    Source Link: {source_url}")
        print(f"    Target Path: {local_dest_path}")
        
        try:
            time.sleep(1.5)  # Safe delay throttle
            
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            with open(local_dest_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)
            
            # --- MUTATE AND RECORD SYNC ---
            manifest = load_manifest()
            manifest["registry"][doc_key]["downloaded"] = True
            manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            save_manifest(manifest)
            
            print(f"    [SUCCESS] Document downloaded safely and entry verified.")
            success_count += 1
            
        except Exception as e:
            print(f"    [DOWNLOAD ERROR] Skipping current item: {e}")
            # Structural Guard: Drop empty or corrupted file fragments if data streaming cuts out
            if os.path.exists(local_dest_path):
                os.remove(local_dest_path)
            continue

    print(f"\n--> Downloader run finished: Successfully pulled {success_count}/{total_queue_count} bills files.")

# Execute downloader function
download_indexed_bills()


                Kenya Gazette

                        1. Indexing the files

In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from urllib.parse import urljoin

# =====================================================================
# GLOBAL SCALING FACTOR (CONTROL PANEL FOR KENYA GAZETTE)
# =====================================================================
# SAMPLE CONFIGURATION
YEARS_TO_CRAWL = [2025, 2026]           # Specific years to target for sampling
LIMIT_PAGES = 2                        # Maximum pagination pages per year
LIMIT_PER_PAGE = 5                     # Maximum files to index per page

# PRODUCTION SCALING CONFIGURATION (Uncomment below when ready to scale)
# YEARS_TO_CRAWL = list(range(2010, 2027)) # Auto-loops from 2010 through 2026
# LIMIT_PAGES = None                      # Crawl every single page
# LIMIT_PER_PAGE = None                   # Crawl every single row

# --- SYSTEM CONFIGURATION ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
CATEGORY = "kenya_gazette"

# Establish explicit folder target paths for file creation
TARGET_DOWNLOAD_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs", CATEGORY)

def load_manifest():
    """Safely loads manifest records from disk using direct string parsing to avoid file pointer collisions."""
    if not os.path.exists(MANIFEST_PATH):
        os.makedirs(DATA_DIR, exist_ok=True)
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
            
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            print("[WARN] Manifest corrupted or malformed. Re-initializing empty system database structure.")
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}

def save_manifest(manifest_data):
    # Resolved the Python Deprecation Warning by migrating to timezone-aware parameters
    manifest_data["system_metadata"]["last_updated"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    manifest_data["system_metadata"]["total_indexed"] = len(manifest_data["registry"])
    manifest_data["system_metadata"]["total_downloaded"] = sum(
        1 for item in manifest_data["registry"].values() if item.get("downloaded")
    )
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2, ensure_ascii=False)

def index_gazettes():
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    print(f"--> Instantiating target folder tree: {TARGET_DOWNLOAD_DIR}")
    os.makedirs(TARGET_DOWNLOAD_DIR, exist_ok=True)
    
    manifest = load_manifest()
    total_indexed_this_run = 0
    
    print(f"--> Initializing Indexer Engine for Category: [{CATEGORY}]")
    print(f"--> Targets: {len(YEARS_TO_CRAWL)} years queued for execution.")
    
    # 1. LOOP THROUGH THE CHRONOLOGICAL YEARS LAYER
    for year in YEARS_TO_CRAWL:
        # Utilizing the valid query layout parameter structure from your workspace
        base_year_url = f"https://kenyalaw.org/gazettes/{year}?q=&sort=-date"
        current_url = base_year_url
        page_counter = 0
        
        print(f"\n=========================================")
        print(f"   PROCESSING GAZETTE REGISTRY YEAR: {year} ")
        print(f"=========================================")
        
        # 2. LOOP THROUGH THE PAGINATION LAYER FOR THE CHOSEN YEAR
        while current_url:
            if LIMIT_PAGES is not None and page_counter >= LIMIT_PAGES:
                print(f"[INFO] Reached maximum page limit ({LIMIT_PAGES}) for year {year}. Advancing timeline.")
                break
                
            page_counter += 1
            print(f"\n--- [YEAR {year}] Reading Table Page {page_counter} ---")
            print(f"Target URL: {current_url}")
            
            try:
                response = requests.get(current_url, headers=headers, timeout=20)
                response.raise_for_status()
            except Exception as e:
                print(f"[ERROR] Connection failure on page {page_counter} for year {year}: {e}")
                break
                
            soup = BeautifulSoup(response.text, "html.parser")
            table_rows = soup.find_all("tr")
            
            if not table_rows:
                print(f"[WARN] No tabular records found on page {page_counter} for year {year}. Breaking page loop.")
                break
                
            items_indexed_on_this_page = 0
            
            # 3. LOOP THROUGH THE DOCUMENT ROWS LAYER
            for row in table_rows:
                if LIMIT_PER_PAGE is not None and items_indexed_on_this_page >= LIMIT_PER_PAGE:
                    print(f" -> Quota of {LIMIT_PER_PAGE} files reached for this page. Shifting window.")
                    break
                    
                # Strict filter using cell-title styling markers to isolate target rows
                title_td = row.find("td", class_="cell-title")
                if not title_td:
                    continue  
                    
                link = title_td.find("a", href=True)
                if not link:
                    continue
                    
                href = link["href"]
                view_url = None
                title_text = ""
                
                # FIX: Explicitly added lowercase, camelCase, and direct relative path keyword checks
                # to catch '/akn/ke/officialGazette/' pathways flawlessly
                if any(x in href.lower() for x in ["/gazette", "/view", "/legislation", "/officialgazette"]):
                    view_url = urljoin(base_year_url, href)
                    title_text = link.get_text(strip=True)
                        
                if not view_url:
                    continue
                    
                if not title_text or any(x in title_text.lower() for x in ["title", "date", "action"]):
                    continue
                    
                # Resilient system key generation splits cleanly across alphanumeric route nodes
                url_clean = view_url.strip("/")
                segments = url_clean.split("/")
                
                if len(segments) >= 2:
                    # Combines trailing tokens like volume/number values securely
                    internal_id = f"{segments[-2]}_{segments[-1]}"
                else:
                    internal_id = str(hash(view_url) % 100000)
                    
                doc_key = f"ke_gazette_{year}_{internal_id}"
                
                # Idempotency Guard
                if doc_key in manifest["registry"]:
                    print(f"   [ALREADY INDEXED] Skipping item: {doc_key}")
                    continue
                    
                print(f"   Found Gazette: '{title_text[:50]}...'")
                download_link = None
                
                # Evaluate landing rules to separate direct links from step-in button pages
                if "/source" in view_url or ".pdf" in view_url.lower():
                    download_link = view_url
                else:
                    print(f"   Stepping into detail view subpage: {view_url}")
                    try:
                        time.sleep(1.0) # Polite rate limiting gap delay
                        sub_response = requests.get(view_url, headers=headers, timeout=15)
                        sub_soup = BeautifulSoup(sub_response.text, "html.parser")
                        
                        # Direct mapping selector matching bootstrap action buttons
                        download_btn = sub_soup.find("a", class_=lambda c: c and "btn-primary" in c and "btn-shrink-sm" in c)
                        if download_btn and download_btn.get("href"):
                            download_link = urljoin(view_url, download_btn["href"])
                        else:
                            for sub_link in sub_soup.find_all("a", href=True):
                                href_lower = sub_link["href"].lower()
                                if "download" in href_lower or ".pdf" in href_lower or "/source" in href_lower:
                                    download_link = urljoin(view_url, sub_link["href"])
                                    break
                                    
                        if not download_link:
                            download_link = view_url
                    except Exception as sub_err:
                        print(f"   [ERROR] Failed to extract subpage info: {sub_err}")
                        continue
                    
                # Target path configuration adjustments
                clean_filename = f"gazette_{year}_{internal_id}.pdf"
                local_rel_path = os.path.join(TARGET_DOWNLOAD_DIR, clean_filename)
                
                # Build entry block mapping schema
                manifest["registry"][doc_key] = {
                    "category": CATEGORY,
                    "year": year,
                    "title": title_text,
                    "source_url": download_link,
                    "local_path": local_rel_path,
                    "downloaded": False,
                    "downloaded_at": None
                }
                
                items_indexed_on_this_page += 1
                total_indexed_this_run += 1
                
            # Commit tracking checkpoints after every table page processing execution
            save_manifest(manifest)
            
            # --- PAGINATION TRAVERSAL WITHIN CURRENT YEAR BLOCK ---
            next_page_url = None
            pagination_block = soup.find("ul", class_="pagination") or soup.find("nav") or soup
            next_link = pagination_block.find("a", rel="next") or \
                        pagination_block.find("a", string=lambda text: text and ("next" in text.lower() or "»" in text))
            
            if next_link and next_link.get("href"):
                next_page_url = urljoin(base_year_url, next_link["href"])
            else:
                for a_tag in soup.find_all("a", href=True):
                    # Correct sequential lookahead traversal using (page_counter + 1)
                    if f"page={page_counter + 1}" in a_tag["href"]:
                        next_page_url = urljoin(base_year_url, a_tag["href"])
                        break
                        
            current_url = next_page_url
            if not current_url:
                print(f"[INFO] No more pages found for year {year}.")
                break
                
    print(f"\n--> Indexing completed. Registered {total_indexed_this_run} gazette entries to data_manifest.json.")

# Trigger the indexer run
index_gazettes()

            


                        2. Downloading all the Kenya Gazette indexed files

In [ ]:
import os
import json
import time
import requests
from datetime import datetime

# --- CONFIGURATION PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
TARGET_CATEGORY = "kenya_gazette"

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        print(f"[ERROR] Manifest file not found at {MANIFEST_PATH}. Run the Gazette indexer first!")
        return None
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_manifest(manifest_data):
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2)

def download_indexed_gazettes():
    manifest = load_manifest()
    if not manifest:
        return

    # Filter out registry queue matching un-downloaded gazettes
    download_queue = [
        (doc_key, item) for doc_key, item in manifest["registry"].items()
        if item["category"] == TARGET_CATEGORY and not item["downloaded"]
    ]
    
    total_queue_count = len(download_queue)
    print(f"--> Found {total_queue_count} pending files to download for category: [{TARGET_CATEGORY}]")
    
    if total_queue_count == 0:
        print("--> No pending files to download. Your Kenya Gazette files match the manifest index completely!")
        return

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    success_count = 0
    
    for index, (doc_key, item) in enumerate(download_queue, start=1):
        title = item["title"]
        source_url = item["source_url"]
        
        # FIX: Directly use the literal path value already recorded in the manifest entries
        local_dest_path = item["local_path"]
        
        # Confirm subdirectory target exists safely on disk prior to writing stream
        parent_dir = os.path.dirname(local_dest_path)
        if parent_dir:
            os.makedirs(parent_dir, exist_ok=True)
        
        print(f"\n[{index}/{total_queue_count}] Processing: {title[:50]}...")
        print(f"    Source Link: {source_url}")
        print(f"    Target Path: {local_dest_path}")
        
        try:
            time.sleep(1.5)  # Dynamic polite sleep
            
            response = requests.get(source_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            with open(local_dest_path, "wb") as pdf_file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        pdf_file.write(chunk)
            
            # --- UPDATE STATUS AND STATE ---
            manifest = load_manifest()
            manifest["registry"][doc_key]["downloaded"] = True
            manifest["registry"][doc_key]["downloaded_at"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            save_manifest(manifest)
            
            print(f"    [SUCCESS] Document downloaded safely and flag toggled.")
            success_count += 1
            
        except Exception as e:
            print(f"    [DOWNLOAD ERROR] Skipping current entry line: {e}")
            # Structural Guard: Drop empty or corrupted file fragments if data streaming cuts out
            if os.path.exists(local_dest_path):
                os.remove(local_dest_path)
            continue

    print(f"\n--> Downloader run finished: Successfully pulled {success_count}/{total_queue_count} gazette files.")

# Execute downloader function
download_indexed_gazettes()


    Parsing, cleaning and storing the data

        Clean all the data e.i the pdfs from scraping folder "scraped_raw_pdfs" and turn them into json format in one single file
        called the full_parsed_scraped_data.json. this file 'full_parsed_scraped_data.json" will be the one used to generate chunks
        that will be vectorised before beeing stored in the chhromadb vector DB called legal_ai_vector_db.

                1. Parsing and Cleaning and saving to consolidated json file

In [ ]:
import os
import re
import json
import fitz  # PyMuPDF
from datetime import datetime

# --- DIRECTORY PATHS ---
DATA_DIR = "data"
MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")

# Your requested single file and folder layout
OUTPUT_DIR = os.path.join(DATA_DIR, "full_parsed_scraped_data")
CONSOLIDATED_JSON_PATH = os.path.join(OUTPUT_DIR, "full_parsed_scraped_data.json")

def load_manifest():
    if not os.path.exists(MANIFEST_PATH):
        raise FileNotFoundError(f"[ERROR] Manifest not found at {MANIFEST_PATH}. Complete the scraping stage first!")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_text_segment(text):
    """Phase 2: Normalize spacing, remove structural clutter, and clean up layouts."""
    # Convert arbitrary multiple spaces/tabs/newlines into a single clean space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_year_fallback(title_text):
    """Fallback helper to extract a 4-digit year from the text title string if missing in manifest."""
    match = re.search(r'\b(19\d{2}|20\d{2})\b', title_text)
    return int(match.group(1)) if match else None

def execute_parse_and_clean_stage():
    # Ensure our target output folder exists physically on your disk drive
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    manifest = load_manifest()
    registry = manifest.get("registry", {})
    
    # This dictionary will hold all processed data across ALL categories
    master_parsed_data = {}
    
    # Process only documents successfully flagged as downloaded
    downloaded_items = {k: v for k, v in registry.items() if v.get("downloaded")}
    
    print(f"--> Starting STEP 1: Parse & Clean Raw PDFs...")
    print(f"--> Found {len(downloaded_items)} total downloaded files across all categories.\n")
    
    for doc_key, meta in downloaded_items.items():
        local_pdf_path = meta["local_path"]
        
        if not os.path.exists(local_pdf_path):
            print(f"[WARN] File missing on disk: {local_pdf_path}. Skipping.")
            continue
            
        print(f"[{meta['category'].upper()}] Processing -> {meta['title'][:50]}...")
        
        document_text_accumulator = []
        
        try:
            with fitz.open(local_pdf_path) as doc:
                for page_num in range(len(doc)):
                    page = doc[page_num]
                    raw_page_text = page.get_text()
                    
                    cleaned_page_text = clean_text_segment(raw_page_text)
                    if cleaned_page_text:
                        document_text_accumulator.append(cleaned_page_text)
                        
            # Join all page texts with a clean space to construct a single full text string
            full_document_text = " ".join(document_text_accumulator)
            
            # FIX: Safely grab the year via .get() or extract it dynamically from the title text
            # This completely avoids throwing a fatal KeyError if 'year' is missing.
            document_year = meta.get("year") or extract_year_fallback(meta["title"])
            
            # Map into our cross-category intermediate master file format
            master_parsed_data[doc_key] = {
                "category": meta["category"],
                "year": document_year,  # Saved safely now
                "title": meta["title"],
                "source_url": meta["source_url"],
                "local_path": local_pdf_path,
                "full_text": full_document_text,
                "processed_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
            }
            
        except Exception as e:
            print(f"   [ERROR] Failed to completely parse document {doc_key}: {e}")
            continue

    # Write out the combined dataset to your single targeted JSON file
    print(f"\n--> Saving all parsed category data into a single master file...")
    with open(CONSOLIDATED_JSON_PATH, "w", encoding="utf-8") as json_out:
        json.dump(master_parsed_data, json_out, indent=2, ensure_ascii=False)
        
    print(f"\n SUCCESS: Unified file generated safely at: {CONSOLIDATED_JSON_PATH}")

# Run Step 1
execute_parse_and_clean_stage()


                2. vectorizing and storing in chromadb vector DB

In [ ]:
import os
import json
import chromadb

# --- DIRECTORY PATHS ---
DATA_DIR = "data"
INPUT_JSON_PATH = os.path.join(DATA_DIR, "full_parsed_scraped_data", "full_parsed_scraped_data.json")
VECTOR_DB_DIR = os.path.join(DATA_DIR, "legal_ai_vector_db")  # Target database path
COLLECTION_NAME = "kenya_legal_documents"

def chunk_text(text, max_chars=1200, overlap=200):
    """Splits text into overlapping chunks to make sure keywords aren't chopped in half at edges."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start += (max_chars - overlap)
    return chunks

def execute_vectorization_and_storage_stage():
    print(f"--> Starting STEP 2: Vectorization & Database Storage...")
    
    if not os.path.exists(INPUT_JSON_PATH):
        raise FileNotFoundError(f"[ERROR] Master JSON file not found at {INPUT_JSON_PATH}. Please run Step 1 first!")
        
    # READ ONLY FROM THE UNIFIED JSON CHECKPOINT FILE
    print(f"--> Loading dataset directly from: {INPUT_JSON_PATH}")
    with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
        master_parsed_data = json.load(f)
        
    print(f"--> Loaded {len(master_parsed_data)} parsed legal documents. Connecting to Vector DB...")
    
    # Initialize the localized persistent database client on disk
    chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
    
    # Establish collection using Cosine distance to measure vector similarity
    collection = chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )
    
    print(f"--> Slicing text and streaming vector payloads to collection: [{COLLECTION_NAME}]")
    
    for doc_key, data in master_parsed_data.items():
        full_text = data.get("full_text", "")
        if not full_text:
            continue
            
        # Segment large legal text strings down into manageable chunks
        text_chunks = chunk_text(full_text, max_chars=1200, overlap=200)
        
        chunk_documents = []
        chunk_metadatas = []
        chunk_ids = []
        
        # FIX: Safely parse year value to avoid type casting crashes down the line
        raw_year = data.get("year")
        try:
            parsed_year = int(raw_year) if raw_year is not None else 0
        except (ValueError, TypeError):
            parsed_year = 0
            
        for idx, chunk in enumerate(text_chunks):
            chunk_documents.append(chunk)
            
            # FIX: Swapped out aggressive bracket lookups for safe `.get()` defaults
            chunk_metadatas.append({
                "doc_key": doc_key,
                "category": str(data.get("category", "unknown")),
                "year": parsed_year,
                "title": str(data.get("title", "Missing Title"))[:200], # Secure boundary limits
                "source_url": str(data.get("source_url", ""))
            })
            chunk_ids.append(f"{doc_key}_chunk_{idx}")
            
        if chunk_documents:
            print(f"   Pushing {len(chunk_documents)} vector chunks for document key: {doc_key}")
            try:
                # ChromaDB automatically vectorizes these chunks using its default model here
                collection.add(
                    documents=chunk_documents,
                    metadatas=chunk_metadatas,
                    ids=chunk_ids
                )
            except Exception as write_err:
                print(f"   [DATABASE ERROR] Failed pushing payloads for {doc_key}: {write_err}")
                continue
            
    print(f"\n=======================================================")
    print(f" SUCCESS: All vectors loaded and indexed inside your DB!")
    print(f" Total Active Database Chunk Records: {collection.count()}")
    print(f" Location: {VECTOR_DB_DIR}")
    print(f"=======================================================")

# Run Step 2
execute_vectorization_and_storage_stage()


    Instructions and DPO dataset generation and Finetuning

        Use gemini ai studio to connect to a gpt model that will generate synthetic dataset for both the instructions finetuning and DPO
        with each having 1500 rows all in json format and keep the json formatted instruction and DPO dataset in a file called sft_finetune_synthetic_data.json and dpo_finetune_synthetic_data.json in the directory finetune_synthetic_data depending on random chunks in the vector DB "legal_ai_vector_db"  and then use a small model like Tinyllama to model the dataset and prepare it to know the basic respose structure.

                Dataset Generation

In [ ]:
import os
import json
import random
import chromadb
import asyncio
from pydantic import BaseModel
from google import genai
from google.genai import types
from google.genai.errors import APIError
from dotenv import load_dotenv


# =====================================================================
# SYNTHETIC DATA GENERATION CONTROL PANEL (SCALING FACTORS)
# =====================================================================
EXPERIMENTAL_MODE = True  # Set to False to scale up to the full 2,000 rows

if EXPERIMENTAL_MODE:
    NUM_CHUNKS_TO_PICK = 10     # Test run: 10 random chunks
    ROWS_PER_CHUNK = 1         # 1 row per chunk -> 10 total rows
    MAX_CONCURRENT_REQUESTS = 1 # Sequential for clean testing
    PACING_DELAY = 1.0          # 1 second between requests
else:
    NUM_CHUNKS_TO_PICK = 500    # Production: 500 diverse legal chunks
    ROWS_PER_CHUNK = 4         # 4 unique user angles per chunk -> 2000 total rows
    MAX_CONCURRENT_REQUESTS = 3 # Safe concurrency layout
    PACING_DELAY = 4.5          # Precise pacing guardrail to structurally stay under RPM/TPM ceilings

# --- PATH CONFIGURATION ---
DATA_DIR = "data"
VECTOR_DB_DIR = os.path.join(DATA_DIR, "legal_ai_vector_db")
COLLECTION_NAME = "kenya_legal_documents"

OUTPUT_DIR = os.path.join(DATA_DIR, "finetune_synthetic_data")
SFT_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "sft_finetune_synthetic_data.json")
DPO_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "dpo_finetune_synthetic_data.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- PYDANTIC VALIDATION SCHEMAS ---
class SFTDatasetRecord(BaseModel):
    instruction: str
    response: str

class DPODatasetRecord(BaseModel):
    prompt: str
    chosen: str
    rejected: str

# =====================================================================
# SYSTEM INSTRUCTIONS
# =====================================================================
SFT_SYSTEM_INSTRUCTION = """You are a warm, highly empathetic Kenyan Legal Interpreter AI. 
Your purpose is to translate dense, intimidating legal jargon into clear, narrative-driven explanations that everyday citizens can immediately relate to and understand.

When a user asks a question, your response MUST strictly adhere to this structural blueprint:
1. THE LAW HEADLINE: State the Bill, Act, or constitutional article in bold ALL CAPS.
2. CONVERSATIONAL SUMMARY: Give a reassuring 1-2 sentence high-level take on what this means for them.
3. NARRATIVE BREAKDOWN BULLETS: Create a simple breakdown using catchy phrases (e.g., 'The Pass-The-Load Effect', 'The Rotten Foundation Rule', 'First In Time Rule') to explain how the law works practically on the ground.
4. THE EXAMPLE STORY: Always include a section starting with 'Imagine you...' that maps the law onto a real-world scenario (like a small kiosk owner, an unexpected heir, a worker, or a citizen) to show exactly how the legal shield or cost plays out.
5. THE ENGAGING INQUIRY: Conclude the response with a friendly follow-up question offering two distinct, practical options for what they might want to look into or do next.

ANTI-HALLUCINATION SHIELD: All constraints, rules, numbers, or rights used must be strictly anchored to the provided text chunk. Do not invent fake laws or outside statutory details.
"""

DPO_SYSTEM_INSTRUCTION = """You are an expert Kenyan Legal Preference Data Engine. 
Your task is to create preference pairs to train our model to reject dry, unhelpful legal text and favor clear, empathetic narrative translations for citizens.

TASKS:
1. 'prompt': Generate a realistic question from a worried or confused citizen regarding how the text chunk affects their life, business, or community.
2. 'chosen': Generate an ideal, beautifully structured narrative response matching the SFT blueprint: Headline -> Conversational Summary -> Narrative Breakdown Bullets -> 'Imagine you...' Story -> Engaging Follow-up Inquiry.
3. 'rejected': Generate a response that fails your standard. Make it fail by doing one of the following:
   - Just copying and pasting dry, cold, confusing legal sections without an explanation.
   - Forgetting to include the 'Imagine you...' narrative story completely.
   - Omitting the engaging follow-up question at the end, leaving the user with nowhere to go.
"""

# =====================================================================
# CORE ENGINE WORKERS WITH DYNAMIC PACING AND ERROR INNER SHIELDS
# =====================================================================
async def generate_sft_row(client, user_prompt_content, parent_metadata, semaphore):
    async with semaphore:
        await asyncio.sleep(PACING_DELAY) # Mandatory structural pacing delay
        retries = 10  # High retry limit so the script refuses to give up
        base_delay = 3.0
        
        # FIX: Safeguard against empty or missing metadata structures safely
        meta_dict = parent_metadata if isinstance(parent_metadata, dict) else {}
        
        for attempt in range(retries):
            try:
                sft_response = await client.aio.models.generate_content(
                    model='gemini-2.5-flash-lite',
                    contents=user_prompt_content,
                    config=types.GenerateContentConfig(
                        system_instruction=SFT_SYSTEM_INSTRUCTION,
                        response_mime_type="application/json",
                        response_schema=SFTDatasetRecord,
                        temperature=0.7
                    ),
                )
                sft_data = json.loads(sft_response.text)
                sft_data["meta_source_doc"] = meta_dict.get("doc_key", "unknown_key")
                sft_data["category"] = meta_dict.get("category", "unknown_category")
                return sft_data
            except APIError as e:
                if e.code == 429: # Quota or limit exception encountered
                    sleep_time = (base_delay ** attempt) + random.uniform(1.0, 3.0)
                    print(f"    [SFT LIMIT HIT] PACING BOUNDARY VIOLATED. Sleeping for {sleep_time:.2f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    print(f"    [SFT API ERROR] {e.code}: {e} | Retrying...")
                    await asyncio.sleep(2)
            except Exception as e:
                print(f"    [SFT unexpected drop]: {e} | Retrying...")
                await asyncio.sleep(2)
        return None

async def generate_dpo_row(client, user_prompt_content, parent_metadata, semaphore):
    async with semaphore:
        await asyncio.sleep(PACING_DELAY) # Mandatory structural pacing delay
        retries = 10
        base_delay = 3.0
        
        # FIX: Safeguard against empty or missing metadata structures safely
        meta_dict = parent_metadata if isinstance(parent_metadata, dict) else {}
        
        for attempt in range(retries):
            try:
                dpo_response = await client.aio.models.generate_content(
                    model='gemini-2.5-flash-lite',
                    contents=user_prompt_content,
                    config=types.GenerateContentConfig(
                        system_instruction=DPO_SYSTEM_INSTRUCTION,
                        response_mime_type="application/json",
                        response_schema=DPODatasetRecord,
                        temperature=0.7
                    ),
                )
                dpo_data = json.loads(dpo_response.text)
                dpo_data["meta_source_doc"] = meta_dict.get("doc_key", "unknown_key")
                dpo_data["category"] = meta_dict.get("category", "unknown_category")
                return dpo_data
            except APIError as e:
                if e.code == 429:
                    sleep_time = (base_delay ** attempt) + random.uniform(1.0, 3.0)
                    print(f"    [DPO LIMIT HIT] PACING BOUNDARY VIOLATED. Sleeping for {sleep_time:.2f}s...")
                    await asyncio.sleep(sleep_time)
                else:
                    print(f"    [DPO API ERROR] {e.code}: {e} | Retrying...")
                    await asyncio.sleep(2)
            except Exception as e:
                print(f"    [DPO unexpected drop]: {e} | Retrying...")
                await asyncio.sleep(2)
        return None

# =====================================================================
# PIPELINE EXECUTION ENGINE
# =====================================================================
async def execute_synthetic_generation_pipeline():
    # 1. Load the variables from your local .env file
    load_dotenv()
    
    # 2. Extract the specific token key from your environment map
    api_key = os.getenv("GEMINI_API_KEY")

    # 3. The SDK automatically detects GEMINI_API_KEY from os.environ, 
    # but passing it directly ensures it works flawlessly.
    client = genai.Client(api_key=api_key)
    
    print(f"--> Connecting to Local Vector DB at: {VECTOR_DB_DIR}")
    chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    
    db_data = collection.get(include=["documents", "metadatas"])
    all_chunks = db_data.get("documents", [])
    all_metadatas = db_data.get("metadatas", [])
    total_db_records = len(all_chunks)
    
    print(f"--> Total database chunks available for sampling: {total_db_records}")
    if total_db_records == 0:
        print("[ERROR] Database is empty! Please populate your ChromaDB collection first.")
        return
        
    sample_pool_size = min(NUM_CHUNKS_TO_PICK, total_db_records)
    random_indices = random.sample(range(total_db_records), sample_pool_size)
    
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    
    sft_accumulator = []
    dpo_accumulator = []
    
    # Load existing checkpoints if resuming after a background interruption
    if os.path.exists(DPO_OUTPUT_PATH) and not EXPERIMENTAL_MODE:
        try:
            with open(DPO_OUTPUT_PATH, "r", encoding="utf-8") as f:
                dpo_accumulator = json.load(f)
            print(f"--> Found existing DPO checkpoint. Loaded {len(dpo_accumulator)} pairs.")
        except: pass

    print(f"--> Strategy: Processing {sample_pool_size} chunks. Multiplier: {ROWS_PER_CHUNK} rows/chunk.")
    
    # Process sequentially block-by-block while executing inner generations concurrently
    for idx, list_index in enumerate(random_indices, start=1):
        context_chunk = all_chunks[list_index]
        parent_metadata = all_metadatas[list_index]
        
        print(f"[{idx}/{sample_pool_size}] Processing Chunk #{list_index}... Generating variants...")
        
        chunk_sft_tasks = []
        chunk_dpo_tasks = []
        
        for sub_row in range(ROWS_PER_CHUNK):
            # FIX: Added dynamic index variation parameters to user prompts.
            # This bypasses LLM cache paths and guarantees 4 unique generation angles.
            user_prompt_content = (
                f"Generate a unique, distinct dataset entry (Variant Option #{sub_row + 1}) "
                f"focusing on a completely different realistic citizen angle based on this text:\n"
                f"---\n{context_chunk}\n---"
            )
            chunk_sft_tasks.append(generate_sft_row(client, user_prompt_content, parent_metadata, semaphore))
            chunk_dpo_tasks.append(generate_dpo_row(client, user_prompt_content, parent_metadata, semaphore))
            
        # Execute current batch concurrently
        sft_results = await asyncio.gather(*chunk_sft_tasks)
        dpo_results = await asyncio.gather(*chunk_dpo_tasks)
        
        # Append successful completions safely
        for r in sft_results:
            if r: sft_accumulator.append(r)
        for r in dpo_results:
            if r: dpo_accumulator.append(r)
            
        # REAL-TIME CHECKPOINT SAVING: Writes progress incrementally
        with open(SFT_OUTPUT_PATH, "w", encoding="utf-8") as sft_file:
            json.dump(sft_accumulator, sft_file, indent=2, ensure_ascii=False)
        with open(DPO_OUTPUT_PATH, "w", encoding="utf-8") as dpo_file:
            json.dump(dpo_accumulator, dpo_file, indent=2, ensure_ascii=False)

    print(f"\n=======================================================")
    print(f" PIPELINE FINISHED COMPLETELY: 100% of tasks completed!")
    print(f" Final SFT Dataset Size: {len(sft_accumulator)} rows saved to {SFT_OUTPUT_PATH}")
    print(f" Final DPO Dataset Size: {len(dpo_accumulator)} pairs saved to {DPO_OUTPUT_PATH}")
    print(f"=======================================================")

# Run execution cleanly inside notebook cells
await execute_synthetic_generation_pipeline()


                Finetuning the model

                        1. Initialize Unsloth and Load the Optimized Base Model

In [ ]:
# import os
# from unsloth import FastLanguageModel
# import torch

# # =====================================================================
# # TRAINING CONTROL PANEL (SCALING FACTORS)
# # =====================================================================
# PRODUCTION_MODE = False  # Set to True when ready to run the full dataset

# if not PRODUCTION_MODE:
#     # Quick trial run parameters
#     MAX_STEPS = 20
#     LOGGING_STEPS = 1
#     SAVE_STEPS = 0
#     NUM_EPOCHS = 1
# else:
#     # Full production scale training parameters
#     MAX_STEPS = 0  # 0 means let it run the full epochs length instead
#     LOGGING_STEPS = 10
#     SAVE_STEPS = 100
#     NUM_EPOCHS = 3

# # Define target architectural limits for a standard T4 environment
# max_seq_length = 2048  # Keeps memory overhead managed
# dtype = None           # Auto-detection (Float16 on T4 accelerators)
# load_in_8bit = False   # Set to True if you hit unexpected VRAM constraints

# # Load the 4-bit optimized variant of Llama 3
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = True, # Critical memory saver on a single T4
# )

# # Apply LoRA/PEFT adapters to dynamically prepare the model layers
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 16, # LoRA Rank (higher takes more memory, 16 is optimal for style adaptation)
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_alpha = 16,
#     lora_dropout = 0, 
#     bias = "none",    
#     use_gradient_checkpointing = "unsloth", # Massive VRAM preservation feature
#     random_state = 3407,
#     use_rslora = False,  
#     loftq_config = None, 
# )
# print("--> Model initialized with optimized PEFT adapters successfully!")

                        2. Format and Map the Dataset to Llama 3 Prompt Formatting

In [ ]:
# from datasets import load_dataset

# # Configuration for file pathing pointers
# DATA_DIR = "data"
# OUTPUT_DIR = os.path.join(DATA_DIR, "finetune_synthetic_data")
# SFT_DATA_PATH = os.path.join(OUTPUT_DIR, "sft_finetune_synthetic_data.json")

# # System instructions to anchor the model persona alignment
# SYSTEM_PROMPT = (
#     "You are a warm, highly empathetic Kenyan Legal Interpreter AI. Your purpose is to "
#     "translate dense, intimidating legal jargon into clear, narrative-driven explanations "
#     "that everyday citizens can immediately relate to and understand."
# )

# # Structure string according to Llama 3 format rules
# llama3_prompt_blueprint = """<|start_header_id|>system<|end_header_id|>

# {}<|eot_id|><|start_header_id|>user<|end_header_id|>

# {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# {}<|eot_id|>"""

# EOS_TOKEN = tokenizer.eos_token # Essential token to ensure model knows when to stop speaking

# def format_dataset_to_llama_style(examples):
#     instructions = examples["instruction"]
#     responses    = examples["response"]
#     texts = []
#     for inst, resp in zip(instructions, responses):
#         # Inject the structured components smoothly
#         formatted_string = llama3_prompt_blueprint.format(SYSTEM_PROMPT, inst, resp) + EOS_TOKEN
#         texts.append(formatted_string)
#     return { "text" : texts }

# # Read your generated synthetic JSON array dataset
# print(f"--> Loading source SFT file from: {SFT_DATA_PATH}")
# raw_dataset = load_dataset("json", data_files=SFT_DATA_PATH, split="train")

# # Slice down for a quick sanity test if not in production mode
# if not PRODUCTION_MODE:
#     raw_dataset = raw_dataset.select(range(min(20, len(raw_dataset))))
#     print(f"--> Experimental Trial Mode: Truncated training sample dataset pool size down to: {len(raw_dataset)}")

# # Execute transformation processing
# formatted_dataset = raw_dataset.map(format_dataset_to_llama_style, batched=True)
# print("--> Dataset mapped into Llama-3 format vectors successfully!")
# print(f"Sample Row Preview:\n{formatted_dataset['text'][0][:350]}...")

                        3. Run the Training Loop with SFTTrainer

In [ ]:
# from trl import SFTTrainer
# from transformers import TrainingArguments

# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = formatted_dataset,
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = False, # Set to False for short instructions to avoid broken structures
#     args = TrainingArguments(
#         per_device_train_batch_size = 2, # Kept light to structurally block out OOM exceptions
#         gradient_accumulation_steps = 4, # Simulates a larger batch size without increasing VRAM load
#         warmup_steps = 5 if not PRODUCTION_MODE else 100,
#         max_steps = MAX_STEPS if MAX_STEPS > 0 else -1,
#         num_train_epochs = NUM_EPOCHS if MAX_STEPS == 0 else 1,
#         learning_rate = 2e-4,
#         fp16 = not torch.cuda.is_preferred_dtype_is_bf16(),
#         bf16 = torch.cuda.is_preferred_dtype_is_bf16(),
#         logging_steps = LOGGING_STEPS,
#         optim = "adamw_8bit", # Massive memory saver optimizer
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         output_dir = "outputs",
#         save_steps = SAVE_STEPS if SAVE_STEPS > 0 else 999999, # Block checkpoints during tiny test runs
#     ),
# )

# print("--> Initializing model optimization loops...")
# trainer_stats = trainer.train()
# print(f"--> Training completed! Total runtime duration metrics: {trainer_stats.metrics['train_runtime']:.2f} seconds.")

                        4. Run a Quick Test Inference to Verify the Persona Alignment

In [ ]:
# # Prepare the model for optimized, fast inference speeds
# FastLanguageModel.for_inference(model)

# test_citizen_question = (
#     "I just heard a rumor that the new Finance Act introduces a higher tax on solar panels "
#     "imported into Kenya. I own a clean energy retail shop in Nakuru. How does this affect me?"
# )

# # Format using the exact same structure used during training
# formatted_input = llama3_prompt_blueprint.format(SYSTEM_PROMPT, test_citizen_question, "")

# inputs = tokenizer([formatted_input], return_tensors = "pt").to("cuda")

# print("\n🚀 Testing Fine-Tuned Assistant Output Persona:\n" + "="*50)
# outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n\n")[-1])
# print("="*50)

                        5. Save the LoRA Model Weights

In [ ]:
# # Save your custom style layers safely to disk
# MODEL_SAVE_PATH = "models/lora_kenya_legal_model"
# model.save_pretrained(MODEL_SAVE_PATH)
# tokenizer.save_pretrained(MODEL_SAVE_PATH)
# print(f"✅ Success! Your custom LoRA layers are saved and ready at '{MODEL_SAVE_PATH}'")

    2. The Sync Scraping

        This scrapes the Kenya Gazette every friday to make sure that the legal ai is upto date and doesn't hallucinate if asked about current
        details. The same processes will be doone here to make sure that the data is updated in the vector database.

In [ ]:

import json
import time
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin
import pypdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

# =====================================================================
# DIRECTORY PATH MATRIX CORRESPONDING EXACTLY TO YOUR BACKEND LAYOUT
# =====================================================================
DATA_DIR = "data"
# FIX: Swapped to point directly to your real core system manifest database file
MASTER_MANIFEST_PATH = os.path.join(DATA_DIR, "data_manifest.json")
SYNC_MANIFEST_PATH = os.path.join(DATA_DIR, "sync_kenya_gazette_manifest.json")
VECTOR_DB_DIR = os.path.join(DATA_DIR, "legal_ai_vector_db")

TARGET_CATEGORY = "kenya_gazette"
# FIX: Aligned explicitly with your established backend folder structure layout
GAZETTE_PDF_DIR = os.path.join(DATA_DIR, "scraped_raw_pdfs", TARGET_CATEGORY)
os.makedirs(GAZETTE_PDF_DIR, exist_ok=True)

# =====================================================================
# CORE PIPELINE SECTIONS
# =====================================================================

def load_master_manifest():
    """Reads the global cross-category database manifest safely."""
    if not os.path.exists(MASTER_MANIFEST_PATH):
        return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
    with open(MASTER_MANIFEST_PATH, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            return {"system_metadata": {"last_updated": "", "total_indexed": 0, "total_downloaded": 0}, "registry": {}}
        return json.loads(content)

def get_new_gazette_entries():
    """Scrapes Kenya Law 2026 portal and identifies fresh delta items using strict layout matching."""
    # FIX: Points directly to active current engine deployment mappings
    base_url = "https://kenyalaw.org"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    
    known_urls = set()
    master_manifest = load_master_manifest()
    
    # Trace URLs indexed globally across all your operations
    for item in master_manifest.get("registry", {}).values():
        if item.get("source_url"):
            known_urls.add(item["source_url"])
            
    # Open or create the specialized weekly tracking sync file
    weekly_history = []
    if os.path.exists(SYNC_MANIFEST_PATH):
        try:
            with open(SYNC_MANIFEST_PATH, "r", encoding="utf-8") as f:
                content = f.read().strip()
                if content:
                    weekly_history = json.loads(content)
                    for entry in weekly_history:
                        known_urls.add(entry.get("url"))
        except Exception as e:
            print(f"--> [Warning] Could not read sync manifest layout: {e}")

    try:
        print(f"--> Scraping chronological feed at: {base_url}")
        response = requests.get(base_url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        delta_items = []
        table_rows = soup.find_all("tr")
        
        for row in table_rows:
            # FIX: Pull only valid text hyperlinks nesting inside cell-title layout slots
            title_td = row.find("td", class_="cell-title")
            if not title_td:
                continue
                
            anchor = title_td.find("a", href=True)
            if not anchor:
                continue
                
            view_url = urljoin(base_url, anchor["href"])
            title_text = anchor.get_text(strip=True)
            
            # Step into the detail layout view if it isn't an exposed PDF asset
            download_link = None
            if "/source" in view_url or ".pdf" in view_url.lower():
                download_link = view_url
            else:
                try:
                    time.sleep(1.0) # Polite spacing barrier
                    sub_res = requests.get(view_url, headers=headers, timeout=15)
                    sub_soup = BeautifulSoup(sub_res.text, "html.parser")
                    download_btn = sub_soup.find("a", class_=lambda c: c and "btn-primary" in c and "btn-shrink-sm" in c)
                    
                    if download_btn and download_btn.get("href"):
                        download_link = urljoin(view_url, download_btn["href"])
                    else:
                        for sub_link in sub_soup.find_all("a", href=True):
                            if "/source" in sub_link["href"].lower():
                                download_link = urljoin(view_url, sub_link["href"])
                                break
                    if not download_link:
                        download_link = view_url
                except Exception as sub_err:
                    print(f"    [WARN] Subpage metadata query drop on {view_url}: {sub_err}")
                    continue
            
            if download_link not in known_urls:
                url_clean = view_url.strip("/")
                segments = url_clean.split("/")
                doc_id = segments[-1] if segments[-1].replace("-", "").isalnum() else str(hash(view_url) % 100000)
                
                delta_items.append({
                    "id": doc_id,
                    "url": download_link,
                    "view_url": view_url,
                    "title": title_text,
                    "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                
        return delta_items, weekly_history, master_manifest
    except Exception as e:
        print(f"❌ Scraping index failure: {e}")
        return [], weekly_history, master_manifest

def download_gazette_pdf(item):
    """Streams the target PDF file straight into your specific raw folder path."""
    local_filename = f"gazette_{item['id']}.pdf"
    destination_path = os.path.join(GAZETTE_PDF_DIR, local_filename)
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    
    try:
        print(f"    Downloading: {item['id']} ...")
        response = requests.get(item['url'], headers=headers, stream=True, timeout=30)
        response.raise_for_status()
        
        with open(destination_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        return destination_path
    except Exception as e:
        print(f"    ❌ Failed downloading {item['id']}: {e}")
        # Clean up structural trash artifacts on crash
        if os.path.exists(destination_path):
            os.remove(destination_path)
        return None

def extract_and_chunk_pdf(file_path, doc_id):
    """Parses raw text layers and prepares overlap chunking for legal retention."""
    try:
        reader = pypdf.PdfReader(file_path)
        full_text = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
                
        compiled_text = "\n".join(full_text)
        if not compiled_text.strip():
            return [], [], []
            
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1400,
            chunk_overlap=250,
            length_function=len
        )
        chunks = text_splitter.split_text(compiled_text)
        
        documents, metadatas, ids = [], [], []
        for index, chunk in enumerate(chunks):
            documents.append(chunk)
            metadatas.append({
                "source_doc_id": doc_id,
                "category": TARGET_CATEGORY,
                "sync_date": datetime.now().strftime("%Y-%m-%d")
            })
            ids.append(f"sync_gazette_{doc_id}_chunk_{index}")
            
        return documents, metadatas, ids
    except Exception as e:
        print(f"    ❌ Text processing/chunking failed for {doc_id}: {e}")
        return [], [], []

# =====================================================================
# PIPELINE EXECUTION HUB (THE FRIDAY RUNNER)
# =====================================================================
async def run_weekly_gazette_synchronizer():
    print("=======================================================")
    print(" STARTED WEEKLY KENYA GAZETTE SYNC ENGINE ")
    print("=======================================================")
    
    # 1. Fetch missing items compared against master & sync histories
    new_items, weekly_history, master_manifest = get_new_gazette_entries()
    
    if not new_items:
        print("\n🎉 Everything is perfectly synchronized! No new items found.")
        print("=======================================================")
        return

    print(f"\n🚀 Detected {len(new_items)} fresh weekly updates to process.")
    
    # 2. Connect directly to your local persistent vector database instance
    chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
    embedding_function = embedding_functions.DefaultEmbeddingFunction()
    
    collection = chroma_client.get_or_create_collection(
        name="kenya_legal_documents", 
        embedding_function=embedding_function
    )
    
    # 3. Process every new delta item sequentially
    for idx, item in enumerate(new_items, start=1):
        print(f"\n[{idx}/{len(new_items)}] Processing item: {item['id']}")
        
        # Download straight to your target folder structure
        downloaded_path = download_gazette_pdf(item)
        if not downloaded_path:
            continue
            
        # Parse and divide text
        docs, metas, ids = extract_and_chunk_pdf(downloaded_path, item['id'])
        
        if docs:
            # Commit vectors to ChromaDB database storage layer
            collection.upsert(documents=docs, metadatas=metas, ids=ids)
            print(f"    ✅ Embedded and upserted {len(docs)} chunks successfully.")
        
        # FIX: Mutate global cross-category data_manifest map state seamlessly
        doc_key = f"ke_gazette_id_{item['id']}"
        master_manifest["registry"][doc_key] = {
            "category": TARGET_CATEGORY,
            "title": item["title"],
            "source_url": item["url"],
            "local_path": downloaded_path,
            "downloaded": True,
            "downloaded_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
        }
        
        # Re-tally global system metadata counts
        master_manifest["system_metadata"]["last_updated"] = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
        master_manifest["system_metadata"]["total_indexed"] = len(master_manifest["registry"])
        master_manifest["system_metadata"]["total_downloaded"] = sum(
            1 for entry in master_manifest["registry"].values() if entry.get("downloaded")
        )
        
        # Save both global and sync tracking files simultaneously after each block run
        with open(MASTER_MANIFEST_PATH, "w", encoding="utf-8") as f:
            json.dump(master_manifest, f, indent=2, ensure_ascii=False)

        # 4. Commit success into the weekly tracking manifest immediately
        weekly_history.append(item)
        with open(SYNC_MANIFEST_PATH, "w", encoding="utf-8") as f:
            json.dump(weekly_history, f, indent=4, ensure_ascii=False)
            
    print("\n=======================================================")
    print(" 🎉 WEEKLY SYNC CONCLUDED COMPLETELY!")
    print(f" Master Manifest synced at: {MASTER_MANIFEST_PATH}")
    print(f" Sync Manifest updated at: {SYNC_MANIFEST_PATH}")
    print("=======================================================")

# Run execution cleanly inside your notebook cell workspace
await run_weekly_gazette_synchronizer()



    3. The RAG System

        Create a system that takes user text and vectorizes it and then looks in the vectorDB "legal_ai_vector_db" for semantic search and then if it gets
        the results it transforms the vectors into texts and then it takes those results to the model so that the model can now translate the legal jargon into understandable and simple language to the user and tracks the conversation of the user to create a history feature for them.

In [2]:
%pip install unsloth

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached unsloth-2026.7.2-py3-none-any.whl.metadata (61 kB)
  Using cached unsloth_zoo-2026.7.2-py3-none-any.whl.metadata (33 kB)
  Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.28.0-cp313-cp313-win_amd64.whl.metadata (5.6 kB)
  Using cached tyro-1.0.15-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-win_amd64.whl.metadata (1.0 kB)
  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached triton_windows-3.7.1.post27-cp313-cp313-win_amd64.whl.metadata (1.8 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-win_amd64.whl.metadata (1.8 kB)
  Using cac

In [ ]:
# %pip install unsloth langchain_text_splitters pypdf

In [ ]:
# %pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
from llama_cpp import Llama  # FIX: Clean, fast CPU-quantized inference engine

# =====================================================================
# CONFIGURATION PANEL
# =====================================================================
DATA_DIR = "data"
VECTOR_DB_DIR = os.path.join(DATA_DIR, "legal_ai_vector_db")
COLLECTION_NAME = "kenya_legal_documents"

# Point directly to your local single extracted GGUF file location
GGUF_MODEL_PATH = os.path.join("models", "unsloth.Q4_K_M.gguf")

print("--> Bootstrapping local fast RAG engine...")

# =====================================================================
# STEP 1: INITIALIZE CHROMADB CLIENT
# =====================================================================
embedding_function = embedding_functions.DefaultEmbeddingFunction()
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = chroma_client.get_collection(
    name=COLLECTION_NAME, 
    embedding_function=embedding_function
)
print(f"✅ Bound to ChromaDB collection: '{COLLECTION_NAME}' ({collection.count()} chunks available)")

# =====================================================================
# STEP 2: LOAD THE COMPILED GGUF VIA LLAMA.CPP (SUPER FAST ON CPU)
# =====================================================================
print(f"--> Loading standalone optimized model from: {GGUF_MODEL_PATH}")
if not os.path.exists(GGUF_MODEL_PATH):
    raise FileNotFoundError(f"[ERROR] GGUF file missing at {GGUF_MODEL_PATH}. Make sure to extract your file!")

# Initialize model directly into RAM. 
# Set n_ctx to match your maximum sequence length allocation bounds
llm = Llama(
    model_path=GGUF_MODEL_PATH,
    n_ctx=2048,
    n_threads=4  # Adjust to match half of your computer's CPU core count for peak performance
)
print("✅ Standalone fine-tuned model successfully initialized into local RAM memory.")

# =====================================================================
# STEP 3: CONSTRUCT RAG EXECUTION LOOP
# =====================================================================
def ask_legal_ai_rag(question: str, top_k: int = 3):
    search_results = collection.query(query_texts=[question], n_results=top_k)
    retrieved_docs = search_results.get("documents", [[]])[0]
    retrieved_metas = search_results.get("metadatas", [[]])[0]
    
    if not retrieved_docs:
        context_str = "No specific background context available from internal database logs."
    else:
        context_str = ""
        for i, (doc, meta) in enumerate(zip(retrieved_docs, retrieved_metas)):
            source_id = meta.get("source_doc_id", "Unknown Gazette")
            context_str += f"[Source Document {source_id} (Chunk {i+1})]\n{doc}\n\n"
            
    SYSTEM_PROMPT = (
        "You are a warm, highly empathetic Kenyan Legal Interpreter AI. Your purpose is to "
        "translate dense, intimidating legal jargon into clear, narrative-driven explanations "
        "that everyday citizens can immediately relate to and understand. Always rely heavily "
        "on the provided context to form your answers."
    )
    
    llama3_prompt_blueprint = """<|start_header_id|>system<|end_header_id|>

{}<|eot_id|><|start_header_id|>user<|end_header_id|>

CONTEXT SOURCE TEXTS:
{}

USER QUESTION:
{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
    formatted_prompt = llama3_prompt_blueprint.format(SYSTEM_PROMPT, context_str, question)
    
    # Execute generation instantly on CPU
    response_payload = llm(
        formatted_prompt,
        max_tokens=512,
        temperature=0.1,
        top_p=0.9,
        stop=["<|eot_id|>", "user", "system"]  # Prevent model from looping or hallucinating roles
    )
    
    clean_response = response_payload["choices"][0]["text"].strip()
    
    print("\n" + "="*80)
    print("🤖 ASSISTANT INTERPRETATION RESPONSE:")
    print("="*80)
    print(clean_response)
    print("="*80 + "\n")

# Run the RAG workflow
ask_legal_ai_rag(
    question="How do the new tax adjustments in the recent gazette notice affect local retail businesses?",
    top_k=3
)


c:\Projects\Learning_Projects\CONSTITUTION_INTERPRETER\constitution-interpretor-model\api\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--> Bootstrapping local RAG variables...
--> Target Execution Device Detected: [CPU]
✅ Bound to ChromaDB collection: 'kenya_legal_documents' (5232 chunks available)
--> Loading tokenizer and causal weights from: models\lora_kenya_legal_model


[huggingface_hub.utils._http|WARNING]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
c:\Projects\Learning_Projects\CONSTITUTION_INTERPRETER\constitution-interpretor-model\api\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\panea\.cache\huggingface\hub\models--unsloth--llama-3-8b-Instruct-bnb-4bit. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode